# Reproducing Silkuniene et al. (2025) from dandisets 001047, 001276, 001277 and 001284

Paper: Silkuniene, Silkunas, Pakhomova, Semenov & Pakhomov (2025), "MHz compression of pulse packets facilitates remote focusing of electroporation", Bioelectrochemistry 166, 109016, doi:10.1016/j.bioelechem.2025.109016. The claims file (`claims/silkuniene-2025.yaml`) was written from the accepted author manuscript in PubMed Central (PMC12331426); the paper has no supplement and links no code.

Datasets, each pinned at its most recent published version (the paper cites no version; "Data will be made available on request"):

- DANDI:001047 version 0.250701.1609. Three versions are published; 0.241211.1856 and 0.250701.1609 hold the same 88 assets, 0.240610.2022 holds 54. The latest is pinned; the notebook shows below that none of its files is from the paper's 15.3-mm array, so the choice among versions does not change what can be tested.
- DANDI:001276 version 0.250701.1609, DANDI:001277 version 0.250701.1610, DANDI:001284 version 0.250701.1610: the only published version of each.

Each file is one stitched image of one well. The files are streamed from the archive and reduced to 26-um blocks as they load; reading takes 20 to 30 minutes.

| Claim | Datasets | Status here |
|---|---|---|
| fig1e-field-simulation | all | blocked: data_missing (no field maps) |
| fig2-heating-below-44c | all | blocked: data_missing (no film images) |
| fig3a, fig3bc, fig3-lorentzian, fig4b-40-fold-ratio | 001047, 001276 | tested on the 001276 wells at 1 Hz, 0.2 MHz and control (claims file amended 2026-11-20); 001047 holds no 15.3-mm array images |
| fig3-tail, fig4b-centre, fig4b-near-electrode, fig4c (2 claims) | 001047 | blocked: data_missing (001047 holds no 15.3-mm array images; no dandiset holds 1-kHz or 10-kHz wells) |
| fig5 (6 claims) | 001284 (+001047) | tested on 001284; fig5c in part |
| fig6 (5 claims) | 001277 | tested |
| fig7 (6 claims) | 001276 (+ others for fig7-sharpest) | tested |

This version answers a reader's two answers of 2026-11-20 (`questions/silkuniene-2025.yaml`). (1) fig3a-centre-enhanced, fig3bc-rate-suppresses-off-target, fig3-lorentzian-profile and fig4b-40-fold-ratio now list 001276, and are tested on its CT26 wells at 2 bursts (x4). (2) The background footprint is an open reading (claims file O1, readings i to iv); every claim whose criterion cites O1 is now also run under each footprint ("footprint readings" cells). Outcomes verified in an earlier round are kept in the record, with any change written as a proposal for a further round.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time, resource
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import cancan as cc

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
T0 = time.time()
print("pinned versions:", cc.VERSIONS)

## Data check

In [ ]:
meta = pd.DataFrame(cc.metadata_table())
print(len(meta), "files")
print(meta.groupby(["dandiset", "cell", "channel"], dropna=False).size().to_string())
print("inter-electrode distance stated in the files (mm):")
print(meta.groupby(["dandiset", "spacing_mm", "first_pulse_kv"], dropna=False).size().to_string())
print("pixel size (um) in the OME metadata:", sorted(meta.pixel_um.dropna().unique()))
meta["extent_mm"] = [tuple(round(s * p / 1000, 1) for s in sh[-2:]) for sh, p in zip(meta["shape"], meta.pixel_um.fillna(cc.PIXEL_UM))]
print(meta.groupby(["dandiset", "extent_mm"]).size().to_string())

DANDI:001047. Every file states a 3.8-mm inter-electrode distance and a 3.2-kV first pulse; the paper's array has 15.3-mm diagonals between 2.8-mm needles (12.8 mm between needle centres along a side) and a 7.2-kV first pulse (Methods 2.2). The CHO files cover 13.0 x 13.8 mm. Two of them are shown, reduced, to check the stated geometry against the images.

In [ ]:
m47 = meta[meta.dandiset == "001047"]
print(m47.groupby(["subject", "channel", "protocol", "packet_rate", "repeats"], dropna=False).size().to_string())
ex47 = [m47[(m47.cell == "CHO") & (m47.channel == "FITC")].iloc[0], m47[(m47.channel == "FITC") & (m47.extent_mm.map(lambda e: e[1] > 25))].iloc[0]]
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for a, r in zip(ax, ex47):
    im = cc.read_reduced(r.asset_id)[0]
    v = im[im > 0]
    a.imshow(np.clip(im, *np.percentile(v, [1, 99.5])), cmap="gray",
             extent=[0, im.shape[1] * cc.MM_PER_PX, im.shape[0] * cc.MM_PER_PX, 0])
    a.set_title(f"001047 {r.subject} {r.channel}, {r.spacing_mm} mm stated"); a.set_xlabel("mm"); a.set_ylabel("mm")
    print(r.path, "extent (mm):", r.extent_mm)
plt.tight_layout(); plt.show()

The images show electrode footprints a few mm apart, two exposure areas side by side in the wide images, as the files describe: these are experiments with a 3.8-mm array, not the paper's 15.3-mm array. The paper's packet-rate series of Figures 3 and 4 (1 Hz to 0.2 MHz on the 15.3-mm array) is not in DANDI:001047.

The other three dandisets hold the 15.3-mm array. The files used below: YO-PRO-1 (FITC) images of 001276, 001277 and 001284, and the propidium and Hoechst images taken 2 h after exposure (two-channel Cy3-DAPI files) of 001284.

In [ ]:
use = meta[((meta.dandiset.isin(["001276", "001277", "001284"])) & (meta.channel == "FITC"))
           | ((meta.dandiset == "001284") & (meta.channel == "Cy3-DAPI"))].copy()
print(len(use), "files to read,", f"{use['size'].sum() / 1e9:.1f} GB")
t = time.time()
imgs = cc.read_many(list(use.asset_id), workers=4)
images = dict(zip(use.asset_id, imgs))
print(f"read in {time.time() - t:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
use["frac_empty"] = [float((images[a][0] == 0).mean()) for a in use.asset_id]
empty = use[use.frac_empty > 0.9]
print("files that are more than 90% empty (zero) pixels, left out:")
print(empty[["dandiset", "path", "channel", "size", "frac_empty"]].to_string())
use = use[use.frac_empty <= 0.9]

In [ ]:
t = time.time()
W = cc.well_profiles(use.to_dict("records"), images, workers=4)
print(f"profiles in {time.time() - t:.0f} s")

def summary_row(o):
    h = o["pa"]["half"]
    return dict(dandiset=o["dandiset"], well=o["subject"], chan=o["chan"], cell=o["cell"],
                protocol=o["protocol"], rate=o["packet_rate"], bursts=o["repeats"], pulse_ns=o["pulse_ns"],
                elev_mm=o["elevation_mm"], sham=o["sham"], geometry=o["how"], n_squares_found=o["n_found"],
                side_mm=o["geom"]["side_mm"], theta=o["geom"]["theta_deg"], bg=o["pa"]["background"],
                c0=h[:, 0].mean(), y6=h[:, 6].mean(), y7=h[:, 7].mean())
S = pd.DataFrame([summary_row(o) for o in W])

Locating the array. The paper measures from the geometrical centre of the four electrodes (Methods 2.5). The footprints are found in each image as dark disks or bright rings of the needle's 2.8-mm diameter at the corners of a square (`cc.find_array`). A square is accepted when each corner scores at least 0.15 and its side is within 0.6 mm of 12.8 mm (standard choices, from inspecting the detections). In many images the footprints are faint (the shading of the stitched tiles is stronger) and with the array raised 1 or 2 mm there are none. For those wells the centre is the centre of the central YP spot, with the square's rotation set to the median of the squares found; this is an assumption, checked below on the wells where both are available.

In [ ]:
F = S[S.chan == "FITC"]
print(F.groupby(["dandiset", "geometry"]).size().to_string())
ok = F[F.geometry == "footprints"]
print(f"squares found: side {ok.side_mm.min():.2f} to {ok.side_mm.max():.2f} mm (median {ok.side_mm.median():.2f}; nominal {cc.SIDE_MM:.2f}); rotation {ok.theta.min():.1f} to {ok.theta.max():.1f} deg (median {ok.theta.median():.1f})")
# where footprints were found, how far is the YP spot centre from the geometric centre?
off = []
for o in W:
    if o["chan"] == "FITC" and o["how"] == "footprints" and o["protocol"] == "CanCan":
        sc = cc.spot_centre(images[o["asset_id"]][0])
        off.append(dict(well=o["subject"], dandiset=o["dandiset"], rate=o["packet_rate"], dx=(sc[0] - o["geom"]["centre"][0]) * cc.MM_PER_PX, dy=(sc[1] - o["geom"]["centre"][1]) * cc.MM_PER_PX))
off = pd.DataFrame(off); off["dist"] = np.hypot(off.dx, off.dy)
print("CANCAN wells with footprints: distance of the YP spot centre from the array centre (mm)")
print(off.groupby("rate").dist.describe().round(2).to_string())
print("wells with the spot more than 1.5 mm off centre:"); print(off[off.dist > 1.5].round(2).to_string())
# 001284: do the FITC and the 2-h images of one well share a frame? Distance between the
# YP spot centre and the propidium spot centre of the same well (0-mm wells, where both spots are strong)
al = []
for o in W:
    if o["dandiset"] == "001284" and o["chan"] == "Cy3" and o["elevation_mm"] == "0":
        fitc = [q for q in W if q["subject"] == o["subject"] and q["chan"] == "FITC"][0]
        a_ = cc.spot_centre(images[fitc["asset_id"]][0]); b_ = cc.spot_centre(images[o["asset_id"]][0])
        al.append(dict(well=o["subject"], shift_mm=float(np.hypot(*np.subtract(a_, b_)) * cc.MM_PER_PX)))
print("YP spot centre against propidium spot centre, same well, 0 mm:", pd.DataFrame(al).round(2).to_dict("records"))

In [ ]:
fig, axs = plt.subplots(8, 16, figsize=(32, 17))
for a in axs.ravel(): a.axis("off")
for a, o in zip(axs.ravel(), [o for o in W if o["chan"] in ("FITC", "Cy3")]):
    im = images[o["asset_id"]][0]; v = im[im > 0]
    a.imshow(np.clip(im, *np.percentile(v, [1, 99.7])), cmap="gray")
    c = o["geom"]["corners"]; a.plot(*np.vstack([c, c[:1]]).T, "r-", lw=0.6); a.plot(*o["geom"]["centre"], "r+")
    a.set_title(f"{o['dandiset'][-3:]} {o['subject'][:2]}{o['subject'][-9:]} {o['chan']} {o['how'][:4]}", fontsize=6)
plt.tight_layout(); plt.show()

In [ ]:
print("wells by dandiset, cell line, protocol, packet rate, bursts (x4), pulse duration and elevation (FITC images):")
print(S[S.chan == "FITC"].groupby(["dandiset", "cell", "protocol", "rate", "bursts", "pulse_ns", "elev_mm", "sham"], dropna=False).size().to_string())
print("\npropidium (Cy3) wells of 001284 by elevation:", S[S.chan == "Cy3"].groupby("elev_mm").size().to_dict())
print("\nbackground (mean of 3 ROIs in the darkest footprint), raw a.u.:")
print(S.groupby(["dandiset", "chan"]).bg.describe().round(0).to_string())
print("\ncentre value after background subtraction, a.u.:")
print(S.groupby(["dandiset", "chan"]).c0.describe().round(0).to_string())
print("\nimage values are uint16 counts, unit 'n.a.' in the files; pixel", cc.PIXEL_UM, "um; blocks of", cc.BLOCK, "pixels =", round(cc.MM_PER_PX * 1000), "um")
print("ROI positions, reading (a):", cc.DIST_A, "mm; reading (b):", np.round(cc.DIST_B, 2), "mm")
conf = meta[meta.dandiset.isin(["001277", "001284"])].groupby(["dandiset", "subject"]).pulse_ns.agg(lambda v: sorted(set(v.dropna())))
print("\nwells whose files state more than one pulse duration:", conf[conf.map(len) > 1].to_dict())
print("001277 files by the pulse duration, packets per burst and packet rate their session descriptions state:")
print(meta[meta.dandiset == "001277"].groupby(["pulse_ns", "packets", "packet_rate"], dropna=False).size().to_string())
dup = meta.groupby(["dandiset", "size"]).filter(lambda g: len(g) > 1)
print("\nfiles of identical size (checked for duplicates below):"); print(dup[["dandiset", "path", "channel", "size"]].to_string())
print("\nOME channel names of the files read, in stored order (the Cy3-DAPI files are read as channel 0 = Cy3, 1 = DAPI):")
print(use.assign(ome=use.ome_channels.map(lambda v: tuple(dict.fromkeys(v)))).groupby(["dandiset", "channel", "ome"]).size().to_string())

Duplicate check: the two Cy3-DAPI files of 001277 well P2-20240925-B3 have the same size. A block of each is compared.

In [ ]:
for _, g in dup.groupby(["dandiset", "size"]):
    a, b = g.asset_id.tolist()[:2]
    fa, da = cc._open(a); fb, db = cc._open(b)
    sl = (0, slice(None), slice(9000, 9300), slice(9000, 9300)) if da["data"].ndim == 4 else (0, slice(9000, 9300), slice(9000, 9300))
    same = np.array_equal(da["data"][sl], db["data"][sl])
    print(g.path.tolist(), "identical block:", same)
    fa.close(); fb.close()

Checks of the description against the data, stated here and taken up in the claims below:

- The pixel size (1.625 um) and the needle diameter agree: the footprints found have the 2.8-mm diameter, and the squares found have sides near the 12.8 mm the paper's geometry gives (printed above). The dandisets' descriptions give a 10-mm inter-electrode distance, which matches no distance of this array (edge to edge along a side is 10.0 mm only if measured between needle centres minus the needle diameter; this is consistent with the data, not shown by it).
- 001277: every file states 9 packets per burst at 0.2 MHz whatever the pulse duration (table above), where the dandiset description and the paper give 18, 9 and 6 packets. One well (P1-20240926-B1) is a sham exposure. The 600-ns group holds 7 wells, where the paper reports n = 5. In one of them (P1-20240926-B2) the YP spot is 5.4 mm from the array centre; another (P1-20240926-B3) has its YP maximum at the centre but more uptake 3 to 5 mm out than any other well of this dandiset (montage above, and measures under fig6d-longer-pulses-sharper-peak).
- 001276: besides the 0.2-MHz burst series it holds CANCAN wells at 1 Hz and control wells, which its description does not mention.
- 001284: the YP groups hold 12, 11 and 11 wells at 0, 1 and 2 mm (paper: n = 6 to 7); the propidium groups hold 5, 5 and 4 (paper: 4 to 5).
- There is no time base to check: each file is a single image. Images of one well taken at different times share one stage frame where this could be checked (001284, above).

## Common definitions

Each experiment (well) gives a profile from its ROIs. ROI positions (claims file O1): reading (a) at whole mm, reading (b) contiguous 0.406-mm steps. Pooling of the diagonals (O2): reading (a) the two diagonals averaged at each signed distance, (b) the four half-diagonals averaged and mirrored. A value "at d mm" is the mean of the four half-diagonals at the ROI nearest d. The paper's graph rule (F8) is `cc.gap_exceeds`. Student's t-tests are two-sided and unpaired (F7).

Background (O1). The claims file, as amended on 2026-11-20 in answer to this attempt's question, names four readings for the footprint that holds the background ROIs: (i) to (iv), the footprint of each electrode at the same position in every well. Here they are the footprints at the upper left, upper right, lower right and lower left of the image (corner order of `cc.find_array`). The darkest footprint of each well, which this attempt used before the answer, is kept as the reference in each claim's first cells; every claim whose criterion cites O1 then has a cell, "footprint readings", that runs its test under each of (i) to (iv). `FP` selects the footprint for `prof` and `at`.

In [ ]:
FP = "min"                  # background footprint: "min" the darkest of the well, 0 to 3 O1 readings (i) to (iv)
FOOTPRINTS = (0, 1, 2, 3)
FP_NAME = {"min": "darkest", 0: "(i) upper left", 1: "(ii) upper right", 2: "(iii) lower right", 3: "(iv) lower left"}

def P(o, roi="a", kind="p"):
    """Profile of a well along the diagonals (kind "p") or parallel to the sides ("q"), with the
    background from the footprint FP."""
    p = o[kind + roi]
    return p if FP == "min" else cc.with_background(p, FP)

def prof(o, roi="a", pool="a", kind="p"):
    return cc.pooled(P(o, roi, kind), pool)

def at(o, d, roi="a"):
    p = P(o, roi)
    i = int(np.argmin(np.abs(p["dists"] - d)))
    return float(p["half"][:, i].mean())

def ms(v):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    return v.mean(), (v.std(ddof=1) / np.sqrt(len(v)) if len(v) > 1 else np.nan), len(v)

def fwhm_each(group, roi, half_range, func="lorentz"):
    out = []
    for o in group:
        x, y = prof(o, roi, "a")
        out.append(cc.fit(x, y, func, half_range)["fwhm"])
    return np.array(out)

def mean_profile(group, roi="a", pool="a", norm=None):
    ys = []
    for o in group:
        x, y = prof(o, roi, pool)
        ys.append(y)
    ys = np.array(ys)
    return x, ys

def wells(dandiset, chan="FITC", **kw):
    out = [o for o in W if o["dandiset"] == dandiset and o["chan"] == chan]
    for k, v in kw.items():
        out = [o for o in out if o[k] == v]
    return out

def at_bg(o, d, bg, roi="a"):
    """As `at`, with the background from one footprint ranked by its value (0 darkest, 3
    brightest) or the mean of the four ("mean")."""
    p = o["pa"] if roi == "a" else o["pb"]
    i = int(np.argmin(np.abs(p["dists"] - d)))
    b = p["backgrounds"].mean() if bg == "mean" else np.sort(p["backgrounds"])[bg]
    return float(p["raw"][:, i].mean() - b)

ROIS, POOLS, RANGES = ("a", "b"), ("a", "b"), (4.0, 5.0, 6.0)

def by_footprint(fn, fps=("min",) + FOOTPRINTS):
    """Rows of fn() under each background footprint, with a column naming it."""
    global FP
    rows = []
    try:
        for fp in fps:
            FP = fp
            rows += [dict(footprint=FP_NAME[fp], **r) for r in fn()]
    finally:
        FP = "min"
    return pd.DataFrame(rows)

def side_vals(group, d, roi="a", pool="a", kind="p"):
    """Per-well values at signed distance d on each side of the pooled profile: two sides under
    pooling (a), one under (b)."""
    out = {}
    for sign in ((-1, 1) if pool == "a" else (1,)):
        vals = []
        for o in group:
            x, y = prof(o, roi, pool, kind)
            vals.append(y[int(np.argmin(np.abs(x - sign * d)))])
        out[sign] = np.array(vals)
    return out

Which wells were located from their footprints and which from the YP spot, group by group, and how much locating from the spot changes the measures. Every CANCAN well whose footprints were found is measured a second time from its YP spot centre with the fixed rotation, and the two are compared. The median FWHM difference is used below to show how the FWHM claims near a tolerance edge depend on this.

In [ ]:
# Per well: how the array was located, the best footprint square found (its weakest corner score,
# side and rotation), the distance of the YP spot centre from the centre used, the centre value and
# the per-experiment FWHM (ROI reading a, 4-mm range). Reviewer's request, round 1.
def group_of(o):
    if o["dandiset"] == "001284": return f"{o['chan']} {o['elevation_mm']} mm"
    if o["dandiset"] == "001277": return "sham" if o["sham"] else f"{o['pulse_ns']} ns"
    return f"{o['cell']} {o['protocol']} {o['packet_rate']} x{o['repeats']}"
rows = []
for o in W:
    if o["chan"] not in ("FITC", "Cy3"): continue
    best = max(o["detections"], key=lambda g: g["score"])
    sc = cc.spot_centre(images[o["asset_id"]][0])
    rows.append(dict(dandiset=o["dandiset"], group=group_of(o), well=o["subject"], geometry=o["how"],
                     best_min_corner=min(best["corner_scores"]), best_side_mm=best["side_mm"], theta_used=o["geom"]["theta_deg"],
                     spot_offset_mm=np.hypot(sc[0] - o["geom"]["centre"][0], sc[1] - o["geom"]["centre"][1]) * cc.MM_PER_PX,
                     c0=at(o, 0), fwhm=cc.fit(*prof(o), "lorentz", 4.0)["fwhm"]))
GEO = pd.DataFrame(rows).sort_values(["dandiset", "group", "geometry", "well"])
print(GEO.round(2).to_string())
print(GEO.groupby(["dandiset", "group", "geometry"]).agg(n=("well", "size"), fwhm=("fwhm", "mean"), c0=("c0", "mean")).round(2).to_string())
# How much does locating the array from the YP spot (with the fixed rotation) change the measures?
# On every CANCAN well whose footprints were found, measure again from the spot centre and compare.
fp_wells = [o for o in W if o["chan"] == "FITC" and o["how"] == "footprints" and o["protocol"] == "CanCan"]
ALT = {}
for o in fp_wells:
    im = images[o["asset_id"]][0]
    g2 = cc.geometry_from_centre(cc.spot_centre(im), cc.THETA_DEFAULT_DEG)
    ALT[o["subject"]] = dict(o, geom=g2, pa=cc.profiles(im, g2, cc.DIST_A), pb=cc.profiles(im, g2, cc.DIST_B))
rows = []
for o in fp_wells:
    q = ALT[o["subject"]]
    r = dict(well=o["subject"], c0_ratio=at(q, 0) / at(o, 0), y6_diff=at(q, 6) - at(o, 6), y7_diff=at(q, 7) - at(o, 7))
    for roi in ROIS:
        for hr in RANGES:
            r[f"dfwhm_{roi}{hr:.0f}"] = cc.fit(*prof(q, roi), "lorentz", hr)["fwhm"] - cc.fit(*prof(o, roi), "lorentz", hr)["fwhm"]
    rows.append(r)
BIAS = pd.DataFrame(rows)
print(f"{len(BIAS)} CANCAN wells measured both ways (spot centre and {cc.THETA_DEFAULT_DEG} deg minus footprints):")
print(BIAS.describe(percentiles=[0.25, 0.5, 0.75]).T.round(3).to_string())
FWHM_BIAS = {(roi, hr): float(BIAS[f"dfwhm_{roi}{hr:.0f}"].median()) for roi in ROIS for hr in RANGES}

## Claims not tested

- fig1e-field-simulation: the simulated field maps (Sim4Life) are in none of the four dandisets; blocked, data_missing.
- fig2-heating-below-44c: the thermochromic film images are in none of the four dandisets; blocked, data_missing.
- fig3-tail-deviation-falls-with-rate, fig4b-centre-no-rate-effect, fig4b-near-electrode-decline, fig4c-fwhm-falls-with-rate, fig4c-log-trend: the claims file lists DANDI:001047 only for these, as they rest on a trend over the packet rates or on the 1-kHz and 10-kHz wells, which two rates cannot test (reader's answer of 2026-11-20). 001047 holds only 3.8-mm array images (above), and no dandiset holds 1-kHz or 10-kHz wells; blocked, data_missing. The tail deviation at the two rates 001276 holds is printed under fig3-lorentzian-profile as a variant that decides nothing.

## Figures 3 and 4: packet rate (DANDI:001276)

In answer to this attempt's question, the reader amended the claims file on 2026-11-20: by the paper the 0.2-MHz condition of the packet-rate series (CT26, 600 ns, 2 bursts (x4); Results 3.1, Fig. 3 legend) is the protocol of the 2-burst CT26 group of Fig. 7, so fig3a-centre-enhanced, fig3bc-rate-suppresses-off-target, fig3-lorentzian-profile and fig4b-40-fold-ratio now list 001276 as well as 001047. 001047 holds none of these wells (data check above). 001276 holds CT26 wells at 2 bursts (x4): CANCAN at 1 Hz (n = 2), CANCAN at 0.2 MHz (n = 6) and controls (n = 4), all three conditions on 13 August 2024; it holds no 1-kHz or 10-kHz wells, so fig3a and fig3-lorentzian, which speak of every rate, are judged on the two rates held and are at most partially reproduced (their criteria). The paper reports n = 3 per rate and n = 5 controls.

The main set is every such well; the wells of 13 August 2024, the one date with all three conditions, are a variant. Readings: ROI positions (O1 a, b), background footprint (O1 i to iv, with the darkest as reference) and pooling (O2 a, b; under reading a each signed side of the profile is taken on its own).

In [ ]:
ct2 = wells("001276", cell="CT26", repeats="2")
print("001276 CT26, 2 bursts (x4), by protocol and rate (darkest footprint as background):")
v = pd.DataFrame([dict(well=o["subject"], protocol=o["protocol"], rate=o["packet_rate"], geometry=o["how"], c0=at(o, 0), y4=at(o, 4), y6=at(o, 6), y7=at(o, 7),
                       **{f"bg_{k}": o["pa"]["backgrounds"][k] for k in FOOTPRINTS}) for o in ct2])
print(v.round(1).to_string())
CTRL, HZ1, MHZ = ("Control", "1Hz"), ("CanCan", "1Hz"), ("CanCan", "0.2MHz")
S1 = {k: [o for o in ct2 if (o["protocol"], o["packet_rate"]) == k] for k in (HZ1, MHZ, CTRL)}
S1_SETS = (("all dates", lambda o: True), ("13 Aug 2024", lambda o: "20240813" in o["subject"]))
print({f"{p} {r}": len(g) for (p, r), g in S1.items()})

### fig3a-centre-enhanced

> Every CANCAN protocol enhanced electroporation at the array centre, while in the control (a single unipolar pulse per packet) YP uptake decreased gradually toward the centre.

For each CANCAN rate, mean YP at x = 0 against mean YP at 4 mm by the graph rule (F8); for the control, mean YP at x = 0 against 4, 6 and 7 mm. Student's t-tests are printed, with the centre of each rate against the control's centre, which the criterion reports.

In [ ]:
def chk_3a():
    rows = []
    for label, sel in S1_SETS:
        G = {k: [o for o in g if sel(o)] for k, g in S1.items()}
        for roi in ROIS:
            for pool in POOLS:
                c = {k: side_vals(g, 0, roi, pool)[1] for k, g in G.items()}
                for sign, ctr4 in side_vals(G[CTRL], 4, roi, pool).items():
                    r = dict(set=label, roi=roi, pool=pool, side=sign)
                    for k in (HZ1, MHZ):
                        y4 = side_vals(G[k], 4, roi, pool)[sign]
                        r[f"{k[1]} c0"] = c[k].mean(); r[f"{k[1]} 4mm"] = y4.mean()
                        r[f"{k[1]} peak"] = bool(c[k].mean() > y4.mean() and cc.gap_exceeds(c[k].mean(), ms(c[k])[1], y4.mean(), ms(y4)[1]))
                        r[f"{k[1]} p c0 vs 4"] = stats.ttest_ind(c[k], y4).pvalue
                        r[f"{k[1]} c0 > ctrl c0"] = bool(c[k].mean() > c[CTRL].mean()); r[f"{k[1]} p vs ctrl c0"] = stats.ttest_ind(c[k], c[CTRL]).pvalue
                    r["ctrl c0"] = c[CTRL].mean()
                    for d in (4, 6, 7):
                        y = side_vals(G[CTRL], d, roi, pool)[sign]
                        r[f"ctrl {d}mm"] = y.mean()
                        r[f"ctrl dip {d}"] = bool(c[CTRL].mean() < y.mean() and cc.gap_exceeds(c[CTRL].mean(), ms(c[CTRL])[1], y.mean(), ms(y)[1]))
                        r[f"ctrl p {d}"] = stats.ttest_ind(c[CTRL], y).pvalue
                    rows.append(r)
    return rows
T3a = by_footprint(chk_3a)
cols = ["footprint", "set", "roi", "pool", "side", "1Hz c0", "1Hz 4mm", "1Hz peak", "0.2MHz c0", "0.2MHz 4mm", "0.2MHz peak", "ctrl c0", "ctrl 4mm", "ctrl 6mm", "ctrl 7mm", "ctrl dip 4", "ctrl dip 6", "ctrl dip 7"]
print(T3a[cols].round(0).to_string())
print("t-test p, ranges over all rows:", {c: (round(T3a[c].min(), 4), round(T3a[c].max(), 4)) for c in T3a if " p " in c or c.startswith("ctrl p")})
for label in ("all dates", "13 Aug 2024"):
    for pool in POOLS:
        q = T3a[(T3a.set == label) & (T3a.footprint != "darkest") & (T3a.pool == pool)]
        print(f"{label}, pooling {pool}: 1 Hz peak in {int(q['1Hz peak'].sum())} of {len(q)}, 0.2 MHz peak in {int(q['0.2MHz peak'].sum())} of {len(q)}")
print("CANCAN centre above the control centre:", {c: int(T3a[c].sum()) for c in T3a if c.endswith("> ctrl c0")}, "of", len(T3a))
for label in ("all dates", "13 Aug 2024"):
    q = T3a[(T3a.set == label) & (T3a.footprint != "darkest")]
    print(f"{label}, readings (i) to (iv) x ROI x pooling x side ({len(q)}): 1 Hz peak in {int(q['1Hz peak'].sum())}, 0.2 MHz peak in {int(q['0.2MHz peak'].sum())}, control below 4, 6 and 7 mm in {int((q['ctrl dip 4'] & q['ctrl dip 6'] & q['ctrl dip 7']).sum())}; control centre {q['ctrl c0'].min():.0f} to {q['ctrl c0'].max():.0f} against 4 mm {q['ctrl 4mm'].min():.0f} to {q['ctrl 4mm'].max():.0f}, 6 mm {q['ctrl 6mm'].min():.0f} to {q['ctrl 6mm'].max():.0f}, 7 mm {q['ctrl 7mm'].min():.0f} to {q['ctrl 7mm'].max():.0f}")

**Result: partially reproduced** (changed from blocked after the third verification round). On the two rates 001276 holds, the CANCAN part holds at 0.2 MHz in every reading and at 1 Hz in most. At 0.2 MHz the centre exceeds YP at 4 mm by a gap larger than the error bars in all 24 combinations of footprint reading (i to iv), ROI reading, pooling and side (t-test p 0.0003 to 0.0053). At 1 Hz (n = 2) it does in 21 of 24: in all 8 under pooling (b) and in 13 of 16 under pooling (a), where one side of the profile fails (t-test p 0.062 to 0.195 over all rows). The control part holds under no reading: the control centre (2237 to 2301 a.u.) is not below YP at 4, 6 and 7 mm by a gap in any of the 24 combinations (YP at 4 mm 1749 to 2267 a.u.; t-test p 0.25 to 0.98 over all rows), and the same holds on the 13 August 2024 wells. The CANCAN part holds, with 3 of 16 readings failing at 1 Hz under pooling (a), and the control part does not; with only two rates held the criterion allows at most partial, so the claim is partially reproduced. Reported: centre YP at 1 Hz is above the control centre in every row (p 0.38 to 0.70), at 0.2 MHz in half (p 0.83 to 0.92).

### fig3bc-rate-suppresses-off-target

> Raising the packet rate from 1 Hz to 0.2 MHz suppressed electroporation better both near the electrodes (diagonal profile) and at the perimeter between electrodes (profile parallel to the sides).

Near the electrodes: YP at 6 and 7 mm on the diagonals. Perimeter: ROIs on the lines through the centre parallel to the sides, which meet the sides 6.4 mm out (`cc.parallel_profiles`, measured on the same images; the dandiset holds the whole-well images, so these ROIs can be placed). The criterion's perimeter readings, (a) the outermost parallel ROIs and (b) the parallel ROIs at 6 mm, are the same ROIs here: 6 mm under whole-mm ROIs and 6.09 mm under contiguous ROIs, the last ROI inside the square. 0.2 MHz against 1 Hz by the graph rule (F8), Student's t-tests printed.

In [ ]:
def chk_3bc():
    rows = []
    for label, sel in S1_SETS:
        G = {k: [o for o in g if sel(o)] for k, g in S1.items()}
        for roi in ROIS:
            for pool in POOLS:
                for where, kind, d in (("diagonal 6 mm", "p", 6), ("diagonal 7 mm", "p", 7), ("perimeter", "q", 7)):
                    hz, mhz = side_vals(G[HZ1], d, roi, pool, kind), side_vals(G[MHZ], d, roi, pool, kind)
                    for sign in hz:
                        a_, b_ = mhz[sign], hz[sign]
                        x = prof(G[MHZ][0], roi, pool, kind)[0]
                        rows.append(dict(set=label, roi=roi, pool=pool, where=where, at_mm=float(np.abs(x[np.argmin(np.abs(x - sign * d))])), side=sign,
                                         mhz=a_.mean(), mhz_se=ms(a_)[1], hz=b_.mean(), hz_se=ms(b_)[1],
                                         gap=bool(a_.mean() < b_.mean() and cc.gap_exceeds(a_.mean(), ms(a_)[1], b_.mean(), ms(b_)[1])), p=stats.ttest_ind(a_, b_).pvalue))
    return rows
T3bc = by_footprint(chk_3bc)
print(T3bc.round(3).to_string())
for label in ("all dates", "13 Aug 2024"):
    q = T3bc[(T3bc.set == label) & (T3bc.footprint != "darkest")]
    print(label, "readings (i) to (iv): 0.2 MHz below 1 Hz by a gap in", {w: f"{int(g.gap.sum())} of {len(g)}" for w, g in q.groupby("where")},
          "; t p", {w: (round(g.p.min(), 4), round(g.p.max(), 4)) for w, g in q.groupby("where")},
          "; by pooling", {pool: f"{int(g.gap.sum())} of {len(g)}" for pool, g in q.groupby("pool")},
          "; 0.2 MHz in % of 1 Hz", {w: (round(100 * (g.mhz / g.hz).min(), 1), round(100 * (g.mhz / g.hz).max(), 1)) for w, g in q.groupby("where")})

**Result: blocked (data_missing), the verified outcome; a change to reproduced, resting on pooling (b), was proposed and the third verification round did not agree.** Over the footprint readings (i) to (iv), YP at 0.2 MHz is 2.5 to 6.6% of that at 1 Hz at 6 mm, 4.3 to 11.9% at 7 mm and 6.8 to 16.4% at the perimeter (t-test p at most 0.0095, 0.0193 and 0.0048). By the graph rule it is below 1 Hz by a gap in 24 of 24 combinations at the perimeter, 23 of 24 at 6 mm and 17 of 24 at 7 mm. Every failure is under pooling (a), on the left side of the profile, where one of the two 1-Hz wells is low: 40 of 48 hold under pooling (a) and 24 of 24 under pooling (b). The readings differ, so the outcome rests on pooling (b), the four half-diagonals averaged, the pooling the Fig. 4A legend gives for the near-electrode values of the same experiments ("measurements in 4 diagonal directions from the center were pooled"); this is an assumption. The 1-Hz group has n = 2 where the paper has n = 3. On the 13 August 2024 wells 39 of 48 hold under pooling (a) and 24 of 24 under (b).

The verifier's view: the Fig. 4A legend describes the Fig. 4B near-electrode values, while this claim cites Fig. 3B, whose legend says "Measurements in two diagonals were averaged and considered as one experiment", plotted from -x to x, as Methods 2.5 does. So the paper supports pooling (a) here. Under pooling (a) the outcome also depends on the footprint (only footprint (i) with whole-mm ROIs passes), which under the claims-file rule gives a block (methods_underspecified), and every t-test gives p of 0.019 or less, so the failures come from the graph rule applied to two 1-Hz wells. The outcome stays as verified, and whether O2 leaves the pooling open for this claim is asked of a reader.

### fig3-lorentzian-profile

> The decline of YP emission from the centre was bell-shaped and fitted a Lorentzian function over -4 to 4 mm nearly perfectly (R > 0.9, p < 0.001) for every packet rate.

The Lorentzian is fitted to the mean diagonal profile and to the mean parallel profile of each rate over -4 to 4 mm. R reading (a): Pearson correlation of fitted and measured values with its t test; reading (b): the square root of R2 with the F test of the fit. Beyond 4 mm, the mean of data minus fit (fig3-tail-deviation-falls-with-rate, blocked) is printed at each rate as a variant that decides nothing.

In [ ]:
def chk_3l():
    rows = []
    for label, sel in S1_SETS:
        G = {k: [o for o in g if sel(o)] for k, g in S1.items()}
        for roi in ROIS:
            for pool in POOLS:
                for kind, direction in (("p", "diagonal"), ("q", "parallel")):
                    for k in (HZ1, MHZ):
                        ys = np.array([prof(o, roi, pool, kind)[1] for o in G[k]]); x = prof(G[k][0], roi, pool, kind)[0]
                        f = cc.fit(x, ys.mean(0), "lorentz", 4.0)
                        far = np.abs(x) > 4
                        dev = (ys.mean(0) - cc.lorentz(x, f["a"], f["w"]))[far].mean()
                        rows.append(dict(set=label, roi=roi, pool=pool, direction=direction, rate=k[1], n_points=f["n"], R_pearson=f["r"], p_pearson=f["p_r"],
                                         R_sqrt_r2=np.sqrt(max(f["r2"], 0)), p_F=f["p_f"], fwhm=f["fwhm"], tail_dev_pct=100 * dev / f["a"]))
    return rows
T3l = by_footprint(chk_3l)
T3l["holds"] = (T3l.R_pearson > 0.9) & (T3l.p_pearson < 0.001) & (T3l.R_sqrt_r2 > 0.9) & (T3l.p_F < 0.001)
print(T3l.to_string(float_format=lambda v: f"{v:.3g}"))
for label in ("all dates", "13 Aug 2024"):
    q = T3l[(T3l.set == label) & (T3l.footprint != "darkest")]
    print(f"{label}, readings (i) to (iv): R > 0.9 with p < 0.001 under both R readings in {int(q.holds.sum())} of {len(q)}; R {min(q.R_pearson.min(), q.R_sqrt_r2.min()):.3f} to {max(q.R_pearson.max(), q.R_sqrt_r2.max()):.3f}; largest p {max(q.p_pearson.max(), q.p_F.max()):.1e}; p below 1e-4 in {int(((q.p_pearson < 1e-4) & (q.p_F < 1e-4)).sum())}")
    print("   failing rows:", q[~q.holds][["footprint", "roi", "pool", "direction", "rate", "R_pearson", "p_pearson", "R_sqrt_r2", "p_F"]].round(4).to_dict("records"))
    print("   variant for fig3-tail (decides nothing), data minus fit beyond 4 mm in % of a:", {f"{d} {r}": (round(g.tail_dev_pct.min(), 1), round(g.tail_dev_pct.max(), 1)) for (d, r), g in q.groupby(["direction", "rate"])})

**Result: partially reproduced** (changed from blocked after the third verification round). At both rates held, 1 Hz and 0.2 MHz, the Lorentzian fitted over -4 to 4 mm to the mean diagonal profile and to the mean parallel profile has R above 0.9 with p below 0.001 under both R readings in all 64 combinations with the footprint readings (i) to (iv): R 0.946 to 0.997, largest p 1.2e-04 (62 of 64 below 1e-4). The 13 August 2024 wells give the same (R 0.946 to 0.994). The claim speaks of every packet rate, and the 1-kHz and 10-kHz wells are in no dandiset, so the criterion judges it on the two rates held and allows at most partial. Variant for fig3-tail-deviation-falls-with-rate (blocked, decides nothing): beyond 4 mm the data lie above the fit by 24.6 to 35.9% of the fitted peak at 1 Hz and 1.1 to 4.4% at 0.2 MHz.

### fig4b-40-fold-ratio

> At 0.2 MHz, YP uptake at the centre was 40 times that near the electrodes, despite the field there being about 4 times weaker (3.3 against about 14 kV/cm).

The six CT26 0.2-MHz wells at 2 bursts (x4). Background-subtracted YP at x = 0 over YP at 6 mm or 7 mm (O5 a, b), as the ratio of group means (i) or the mean of per-well ratios (ii), under the ROI readings (O1 a, b), each background footprint (O1 i to iv) and pooling (O2 a, each side; O2 b). The criterion: reproduced when the ratio is in 36 to 44 under one combination, the outcome resting on the footprint that gives it. The fields are from the paper's simulation and are not tested.

In [ ]:
def chk_4b():
    rows = []
    for label, sel in S1_SETS:
        g = [o for o in S1[MHZ] if sel(o)]
        for roi in ROIS:
            for pool in POOLS:
                c0 = side_vals(g, 0, roi, pool)[1]
                for d in (6, 7):
                    for sign, y in side_vals(g, d, roi, pool).items():
                        rows.append(dict(set=label, n=len(g), roi=roi, pool=pool, dist=d, side=sign, centre=c0.mean(), near=y.mean(), near_se=ms(y)[1],
                                         ratio_of_means=c0.mean() / y.mean(), mean_of_ratios=np.mean(c0 / y), min_near_well=y.min()))
    return rows
T4b = by_footprint(chk_4b)
T4b["in_i"] = T4b.ratio_of_means.between(36, 44); T4b["in_ii"] = T4b.mean_of_ratios.between(36, 44)
print(T4b.round(2).to_string())
for label in ("all dates", "13 Aug 2024"):
    q = T4b[(T4b.set == label) & (T4b.footprint != "darkest")]
    print(f"{label}, readings (i) to (iv): combinations inside 36 to 44: {int((q.in_i | q.in_ii).sum())} of {len(q)} rows")
    print("   ", q[q.in_i | q.in_ii][["footprint", "roi", "pool", "dist", "side", "ratio_of_means", "mean_of_ratios"]].round(1).to_dict("records"))
    for fp, h in q.groupby("footprint"):
        print(f"   {fp}: 6 mm ratio of means {h[h.dist == 6].ratio_of_means.min():.1f} to {h[h.dist == 6].ratio_of_means.max():.1f}, mean of ratios {h[h.dist == 6].mean_of_ratios.min():.1f} to {h[h.dist == 6].mean_of_ratios.max():.1f}; 7 mm {h[h.dist == 7].ratio_of_means.min():.1f} to {h[h.dist == 7].ratio_of_means.max():.1f} and {h[h.dist == 7].mean_of_ratios.min():.1f} to {h[h.dist == 7].mean_of_ratios.max():.1f}")

**Result: reproduced, resting on an assumption** (changed from blocked after the third verification round). With the six 0.2-MHz wells, the centre to near-electrode ratio is inside 36 to 44 in 10 of the 48 combinations of footprint reading (i to iv), ROI reading, pooling, side and distance, all at 6 mm. The first combination the criterion lists, footprint (i), pooling (a), 6 mm, ratio of means, with whole-mm ROIs, gives 39.4 on the left side and 33.4 on the right; under pooling (b), footprint (iii) at 6 mm gives 43.9 and 43.2 over the ROI readings. At 6 mm the ratio of means is 33.4 to 39.4 with footprint (i), 31.2 to 36.3 with (ii), 39.9 to 48.8 with (iii) and 75.0 to 114.9 with (iv); at 7 mm every footprint gives 10.6 to 29.5. The mean of per-well ratios is unstable, as with a fixed footprint some wells' near-electrode values fall below zero. As the criterion says, the outcome rests on the footprint that gives the value; the darkest footprint, the reference, gives a 6-mm ratio of means of 26.7 to 30.4 and a mean of ratios of 35.4 to 52.0. On the 13 August 2024 wells (n = 3), 6 of 48 combinations are inside. The evidence is weak: the footprint alone moves the 6-mm ratio of means across the whole 36 to 44 band, and the darkest footprint gives no ratio of means in range. The fields are the paper's simulation and are not tested.

## Figure 5: raised electrodes and cell killing (DANDI:001284)

### fig5-raised-electrodes-weaken-centre

> Since with MHz-CANCAN there was practically no YP staining near electrodes (a finding the text cites to Figure 4), the only clear effect of raising the electrodes 1 or 2 mm above the monolayer was a weakening of electroporation in the central spot.

Centre YP in % of the mean 0-mm centre (Fig. 5B legend); 1 and 2 mm against 0 mm by the graph rule (F8), with Student's t-tests, Bonferroni over the two. The centre ROI is the same under every reading of O1 and O2.

In [ ]:
yp = {e: wells("001284", elevation_mm=e) for e in ("0", "1", "2")}
c0ref = np.mean([at(o, 0) for o in yp["0"]])
rows = []
for e, g in yp.items():
    c = np.array([at(o, 0) for o in g]) / c0ref * 100
    m, se, n = ms(c)
    rows.append(dict(elev=e, n=n, centre_pct=m, se=se, y6_pct=ms([at(o, 6) / c0ref * 100 for o in g])[0], y7_pct=ms([at(o, 7) / c0ref * 100 for o in g])[0]))
R5 = pd.DataFrame(rows); print(R5.round(2).to_string())
c = {e: np.array([at(o, 0) for o in g]) / c0ref * 100 for e, g in yp.items()}
for e in ("1", "2"):
    t = stats.ttest_ind(c[e], c["0"])
    print(f"{e} mm vs 0 mm: gap rule {cc.gap_exceeds(c['0'].mean(), ms(c['0'])[1], c[e].mean(), ms(c[e])[1])}; t = {t.statistic:.2f}, p = {t.pvalue:.4f}, Bonferroni x2 p = {min(1, 2 * t.pvalue):.4f}")
# by batch of experiments (dates), as the dandiset holds more wells than the paper reports
for batch, sel in (("Aug 2024", lambda o: "202408" in o["subject"]), ("Oct 2024", lambda o: "202410" in o["subject"])):
    print(batch, {e: (round(np.mean([at(o, 0) / c0ref * 100 for o in g if sel(o)]), 1), sum(sel(o) for o in g)) for e, g in yp.items()})

In [ ]:
# YP wells of the August 2024 batch only (7, 6 and 7 wells, the paper's n = 6 to 7)
aug = {e: [o for o in g if "202408" in o["subject"]] for e, g in yp.items()}
print("August wells per elevation:", {e: len(g) for e, g in aug.items()}, "; October:", {e: len(g) - len(aug[e]) for e, g in yp.items()})
c0aug = np.mean([at(o, 0) for o in aug["0"]])
ca = {e: np.array([at(o, 0) for o in g]) / c0aug * 100 for e, g in aug.items()}
for e in ("0", "1", "2"):
    m, se, n = ms(ca[e])
    print(f"August, {e} mm: centre {m:.1f} +/- {se:.1f}% of the August 0-mm mean (n={n}); 6 mm {ms([at(o, 6) / c0aug * 100 for o in aug[e]])[0]:.1f}%, 7 mm {ms([at(o, 7) / c0aug * 100 for o in aug[e]])[0]:.1f}%")
for e in ("1", "2"):
    t = stats.ttest_ind(ca[e], ca["0"])
    print(f"August, {e} mm vs 0 mm: gap rule {cc.gap_exceeds(ca['0'].mean(), ms(ca['0'])[1], ca[e].mean(), ms(ca[e])[1])}; t = {t.statistic:.2f}, p = {t.pvalue:.4f}, Bonferroni x2 p = {min(1, 2 * t.pvalue):.4f}")

In [ ]:
# Footprint readings (O1 i to iv, claims file as amended 2026-11-20), with the darkest footprint as reference
def chk_5r():
    rows = []
    for label, grp in (("all wells", yp), ("August", aug)):
        ref = np.mean([at(o, 0) for o in grp["0"]])
        c = {e: np.array([at(o, 0) for o in g]) / ref * 100 for e, g in grp.items()}
        r = dict(set=label)
        for e in ("1", "2"):
            r[f"centre_{e}mm_pct"] = c[e].mean(); r[f"se_{e}"] = ms(c[e])[1]
            r[f"gap_{e}"] = bool(c[e].mean() < c["0"].mean() and cc.gap_exceeds(c["0"].mean(), ms(c["0"])[1], c[e].mean(), ms(c[e])[1]))
            r[f"p_bonf_{e}"] = min(1, 2 * stats.ttest_ind(c[e], c["0"]).pvalue)
        r["se_0"] = ms(c["0"])[1]; r["near_pct_max"] = max(np.mean([at(o, d) for o in g]) / ref * 100 for g in grp.values() for d in (6, 7))
        r["holds"] = r["gap_1"] and r["gap_2"]
        rows.append(r)
    return rows
T = by_footprint(chk_5r); print(T.round(3).to_string())

**Result: not reproduced.** With all wells, raising the electrodes lowered mean centre YP to 48.7% (1 mm, n = 11) and 28.9% (2 mm, n = 11) of the 0-mm value (n = 12). Only the 2-mm drop is larger than the error bars by the paper's graph rule; the 1-mm drop is not (t-test p = 0.079, Bonferroni 0.158; 2 mm p = 0.028, Bonferroni 0.056). With the August 2024 wells alone (7, 6 and 7 wells, the paper's n = 6 to 7), the centre falls to 68.2% and 49.0% of the August 0-mm mean and neither drop is larger than the error bars (Bonferroni p = 0.558 and 0.356). The criterion needs both drops, which neither set of wells gives. YP at 6 and 7 mm is 1.5 to 7.4% of the 0-mm centre at every elevation with all wells, and 2.0 to 12.7% in the August wells. In the October 2024 wells alone the centre falls from 121.8% to 38.0% and 7.1%.

Under the footprint readings (i) to (iv) of the amended O1 the outcome is the same: with all wells the 1-mm centre is 48.2 to 48.5% of the 0-mm mean (no gap) and the 2-mm centre 28.2 to 28.5% (gap), and with the August wells neither drop exceeds the error bars.

### fig5-killing-focused-centre

> With MHz-CANCAN, cell killing (propidium uptake 2 h after exposure) was produced at the centre of the array with negligible effect near the electrodes.

Pr at x = 0 against 6 and 7 mm at 0 mm elevation, graph rule and Student's t-tests; a paired t-test (same wells) as a variant.

In [ ]:
pr = {e: wells("001284", "Cy3", elevation_mm=e) for e in ("0", "1", "2")}
for e, g in pr.items():
    for roi in ROIS:
        c0 = np.array([at(o, 0, roi) for o in g]); y6 = np.array([at(o, 6, roi) for o in g]); y7 = np.array([at(o, 7, roi) for o in g])
        m0, s0, n = ms(c0)
        line = f"elev {e} mm, ROI reading {roi}, n={n}: centre {m0:.0f} +/- {s0:.0f}"
        for d, y in ((6, y6), (7, y7)):
            m, s, _ = ms(y)
            t = stats.ttest_ind(c0, y); tp = stats.ttest_rel(c0, y)
            line += f"; {d} mm {m:.1f} +/- {s:.1f} ({100 * m / m0:.1f}% of centre), gap {cc.gap_exceeds(m0, s0, m, s)}, t p={t.pvalue:.4f}, paired p={tp.pvalue:.4f}"
        print(line)

In [ ]:
# Footprint readings (O1 i to iv, claims file as amended 2026-11-20), with the darkest footprint as reference
def chk_5k():
    rows = []
    g = pr["0"]
    for roi in ROIS:
        c0 = np.array([at(o, 0, roi) for o in g]); m0, s0, _ = ms(c0)
        r = dict(roi=roi, centre=m0, centre_se=s0)
        for d in (6, 7):
            y = np.array([at(o, d, roi) for o in g]); m, s, _ = ms(y)
            r[f"pr_{d}"] = m; r[f"pct_{d}"] = 100 * m / m0; r[f"gap_{d}"] = bool(m0 > m and cc.gap_exceeds(m0, s0, m, s)); r[f"p_{d}"] = stats.ttest_ind(c0, y).pvalue
        r["holds"] = r["gap_6"] and r["gap_7"]
        rows.append(r)
    return rows
T = by_footprint(chk_5k); print(T.round(4).to_string())

**Result: reproduced.** At 0 mm (n = 5), centre Pr is 1188 +/- 339 a.u. and Pr at 6 and 7 mm is 8.2 to 16.5 a.u. (0.7 to 1.4% of the centre), below the centre by more than the error bars under both ROI readings (Student's t-test p = 0.008 to 0.009; paired p = 0.025). At 1 mm Pr at 6 and 7 mm is about 13% of the centre and the gap holds; at 2 mm the centre Pr is 23 a.u. and it does not.

Under the footprint readings (i) to (iv) the gaps hold under both ROI readings, with Pr at 6 and 7 mm -0.2 to 1.2% of the centre (t-test p 0.008 to 0.009).

### fig5ab-yp-spot-slightly-larger-than-pr

> The central spot of YP staining (electroporation) was only slightly larger than the spot of Pr staining (cell killing).

FWHM of the Lorentzian fit to the mean YP profile and of the Gaussian fit to the mean Pr profile, -5 to 5 mm, in % of the 0-mm peak (Fig. 5B legend); Gaussian FWHM 2.4 sigma (Methods) and 2.355 sigma (variant). Ratio judged against 1 to 1.25 (reading a) and 1 to 1.5 (reading b). The YP wells of 0 mm are all 12 (main), and the 5 wells that also have a Pr image as a variant.

In [ ]:
def pct_profile(group, roi, pool, ref):
    x, ys = mean_profile(group, roi, pool)
    return x, ys / ref * 100

rows = []
for e in ("0", "1", "2"):
    for roi in ROIS:
        for pool in POOLS:
            for ypset, ypg in (("all YP wells", yp[e]), ("wells with Pr", [o for o in yp[e] if o["subject"] in {p["subject"] for p in pr[e]}])):
                xr, ysy = pct_profile(ypg, roi, pool, np.mean([at(o, 0, roi) for o in yp["0"]]))
                xp, ysp = pct_profile(pr[e], roi, pool, np.mean([at(o, 0, roi) for o in pr["0"]]))
                fy = cc.fit(xr, ysy.mean(0), "lorentz", 5.0); fp = cc.fit(xp, ysp.mean(0), "gauss", 5.0)
                rows.append(dict(elev=e, roi=roi, pool=pool, yp_set=ypset, n_yp=len(ypg), n_pr=len(pr[e]), fwhm_yp=fy["fwhm"], fwhm_pr=fp["fwhm"], ratio=fy["fwhm"] / fp["fwhm"], ratio_2355=fy["fwhm"] / (fp["w"] * 2.355)))
R5ab = pd.DataFrame(rows); print(R5ab.round(3).to_string())
z = R5ab[(R5ab.elev == "0") & (R5ab.yp_set == "all YP wells")]
print(f"0 mm, all YP wells: ratio {z.ratio.min():.3f} to {z.ratio.max():.3f}; within 1-1.25 under every reading: {bool(((z.ratio >= 1) & (z.ratio <= 1.25)).all())}; within 1-1.5: {bool(((z.ratio >= 1) & (z.ratio <= 1.5)).all())}")
z2 = R5ab[(R5ab.elev == "0") & (R5ab.yp_set == "wells with Pr")]
print(f"0 mm, wells with Pr: ratio {z2.ratio.min():.3f} to {z2.ratio.max():.3f}")

In [ ]:
# Footprint readings (O1 i to iv, claims file as amended 2026-11-20), with the darkest footprint as reference
def chk_5ab():
    rows = []
    for ypset, ypg in (("all YP wells", yp["0"]), ("August YP wells", aug["0"])):
        for roi in ROIS:
            for pool in POOLS:
                xr, ysy = pct_profile(ypg, roi, pool, np.mean([at(o, 0, roi) for o in ypg]))
                xp, ysp = pct_profile(pr["0"], roi, pool, np.mean([at(o, 0, roi) for o in pr["0"]]))
                fy = cc.fit(xr, ysy.mean(0), "lorentz", 5.0); fp = cc.fit(xp, ysp.mean(0), "gauss", 5.0)
                rows.append(dict(yp_set=ypset, roi=roi, pool=pool, fwhm_yp=fy["fwhm"], fwhm_pr=fp["fwhm"], ratio=fy["fwhm"] / fp["fwhm"], ratio_2355=fy["fwhm"] / (fp["w"] * 2.355)))
    return rows
T = by_footprint(chk_5ab); print(T.round(3).to_string())
q = T[(T.yp_set == "all YP wells") & (T.footprint != "darkest")]
print(f"all YP wells, readings (i) to (iv): ratio {q.ratio.min():.3f} to {q.ratio.max():.3f}; within 1 to 1.25 in {int(q.ratio.between(1, 1.25).sum())} and within 1 to 1.5 in {int(q.ratio.between(1, 1.5).sum())} of {len(q)}")

**Result: not reproduced.** At 0 mm the FWHM of the Lorentzian fit to mean YP is 1.69 to 1.75 mm and that of the Gaussian fit to mean Pr 2.29 to 2.34 mm, a ratio of 0.74 to 0.75 under every reading (0.66 to 0.67 with only the 5 wells that have a Pr image; 0.75 to 0.76 with FWHM = 2.355 sigma). By FWHM the YP spot is smaller than the Pr spot, outside 1 to 1.25 and 1 to 1.5. With the 7 August YP wells as the YP group (the paper's n, printed under fig5b-gaussian-pr-lorentzian-yp) the ratio is 0.82 to 0.83.

Under the footprint readings (i) to (iv) the ratio is 0.723 to 0.750 with all YP wells, outside both ranges in all 16 combinations.

### fig5b-gaussian-pr-lorentzian-yp

> Pr intensity declined from the peak as a Gaussian and YP as a Lorentzian; in the left panel of Figure 5B the R2 was 0.92 (Lorentzian) against 0.96 (Gaussian) for Pr, and 0.98 against 0.95 for YP, fitted from -5 to 5 mm with p < 0.0001.

R2 = 1 - SSres/SStot of each fit to the mean 0-mm profile in %, -5 to 5 mm; the F test of the fit is printed.

In [ ]:
rows = []
for roi in ROIS:
    for pool in POOLS:
        xr, ysy = pct_profile(yp["0"], roi, pool, np.mean([at(o, 0, roi) for o in yp["0"]]))
        xp, ysp = pct_profile(pr["0"], roi, pool, np.mean([at(o, 0, roi) for o in pr["0"]]))
        r = dict(roi=roi, pool=pool)
        for name, (x, y) in (("pr", (xp, ysp.mean(0))), ("yp", (xr, ysy.mean(0)))):
            for func in ("lorentz", "gauss"):
                f = cc.fit(x, y, func, 5.0)
                r[f"{name}_{func}_r2"] = f["r2"]; r[f"{name}_{func}_p"] = f["p_f"]; r[f"{name}_n"] = f["n"]
        rows.append(r)
R5b = pd.DataFrame(rows); print(R5b.to_string(float_format=lambda v: f"{v:.3g}"))
exp = dict(pr_lorentz_r2=0.92, pr_gauss_r2=0.96, yp_lorentz_r2=0.98, yp_gauss_r2=0.95)
for k, v in exp.items():
    print(f"{k}: expected {v} +/- {0.1 * v:.3f}; measured {R5b[k].min():.3f} to {R5b[k].max():.3f}; within tolerance under every reading: {bool((abs(R5b[k] - v) <= 0.1 * v).all())}")
print("Pr: Gaussian above Lorentzian under every reading:", bool((R5b.pr_gauss_r2 > R5b.pr_lorentz_r2).all()))
print("YP: Lorentzian above Gaussian under every reading:", bool((R5b.yp_lorentz_r2 > R5b.yp_gauss_r2).all()))

In [ ]:
# fig5ab and fig5b with the August YP wells (n = 7 at 0 mm) against the five Pr wells
rows = []
for roi in ROIS:
    for pool in POOLS:
        xr, ysy = pct_profile(aug["0"], roi, pool, np.mean([at(o, 0, roi) for o in aug["0"]]))
        xp, ysp = pct_profile(pr["0"], roi, pool, np.mean([at(o, 0, roi) for o in pr["0"]]))
        fy = cc.fit(xr, ysy.mean(0), "lorentz", 5.0); fp = cc.fit(xp, ysp.mean(0), "gauss", 5.0); fyg = cc.fit(xr, ysy.mean(0), "gauss", 5.0)
        rows.append(dict(roi=roi, pool=pool, n_yp=len(aug["0"]), fwhm_yp=fy["fwhm"], fwhm_pr=fp["fwhm"], ratio=fy["fwhm"] / fp["fwhm"], ratio_2355=fy["fwhm"] / (fp["w"] * 2.355), yp_lorentz_r2=fy["r2"], yp_gauss_r2=fyg["r2"], yp_lorentz_p=fy["p_f"], yp_gauss_p=fyg["p_f"]))
R5aug = pd.DataFrame(rows); print(R5aug.to_string(float_format=lambda v: f"{v:.3g}"))
print(f"August YP wells: FWHM ratio {R5aug.ratio.min():.3f} to {R5aug.ratio.max():.3f}; YP R2 Lorentzian {R5aug.yp_lorentz_r2.min():.3f} to {R5aug.yp_lorentz_r2.max():.3f}, Gaussian {R5aug.yp_gauss_r2.min():.3f} to {R5aug.yp_gauss_r2.max():.3f}; Lorentzian above Gaussian under every reading: {bool((R5aug.yp_lorentz_r2 > R5aug.yp_gauss_r2).all())}")

In [ ]:
# Footprint readings (O1 i to iv, claims file as amended 2026-11-20), with the darkest footprint as reference
def chk_5b():
    rows = []
    for roi in ROIS:
        for pool in POOLS:
            xr, ysy = pct_profile(yp["0"], roi, pool, np.mean([at(o, 0, roi) for o in yp["0"]]))
            xp, ysp = pct_profile(pr["0"], roi, pool, np.mean([at(o, 0, roi) for o in pr["0"]]))
            r = dict(roi=roi, pool=pool)
            for name, (x, y) in (("pr", (xp, ysp.mean(0))), ("yp", (xr, ysy.mean(0)))):
                for func in ("lorentz", "gauss"):
                    f = cc.fit(x, y, func, 5.0); r[f"{name}_{func}_r2"] = f["r2"]; r[f"{name}_{func}_p"] = f["p_f"]
            rows.append(r)
    return rows
T = by_footprint(chk_5b); print(T.to_string(float_format=lambda v: f"{v:.3g}"))
q = T[T.footprint != "darkest"]
inside = all((abs(q[k] - v) <= 0.1 * v).all() for k, v in exp.items())
print(f"readings (i) to (iv): every R2 within tolerance {inside}; Pr Gaussian above Lorentzian {bool((q.pr_gauss_r2 > q.pr_lorentz_r2).all())}; YP Lorentzian above Gaussian {bool((q.yp_lorentz_r2 > q.yp_gauss_r2).all())}; largest F-test p {q[[c for c in q if c.endswith('_p')]].max().max():.1e}")
print({k: (round(q[k].min(), 3), round(q[k].max(), 3)) for k in exp})

**Result: reproduced.** R2 of the Lorentzian and Gaussian fits are 0.937 to 0.960 and 0.984 to 0.998 for Pr, and 0.975 to 0.993 and 0.960 to 0.979 for YP, each within 10% of 0.92, 0.96, 0.98 and 0.95, with the Gaussian better for Pr and the Lorentzian better for YP under every reading. Every fit's F test has p below 0.0001. With the 7 August YP wells, YP R2 is 0.976 to 0.995 (Lorentzian) and 0.947 to 0.966 (Gaussian), the Lorentzian above under every reading.

Under the footprint readings (i) to (iv), R2 is 0.933 to 0.959 (Pr, Lorentzian), 0.984 to 0.998 (Pr, Gaussian), 0.973 to 0.993 (YP, Lorentzian) and 0.962 to 0.984 (YP, Gaussian), all within tolerance with both orderings holding, and the largest F-test p is 1.4e-06.

### fig5b-reversible-beyond-2mm

> The Lorentzian YP profile and the Gaussian Pr profile differ mainly in the tails, where YP is broader, marking an area of predominantly reversible electroporation that starts about 2 mm from the centre.

YP and Pr each in % of its own 0-mm peak. Reading (a): the smallest distance at which mean YP% exceeds mean Pr% by a gap larger than the error bars; reading (b): the smallest distance at which the fitted Lorentzian (in % of its a) exceeds the fitted Gaussian (in % of its a) by more than the mean s.e. of YP%. Each side separately (reading a of O2), then averaged.

In [ ]:
rows = []
for roi in ROIS:
    xr, ysy = pct_profile(yp["0"], roi, "a", np.mean([at(o, 0, roi) for o in yp["0"]]))
    xp, ysp = pct_profile(pr["0"], roi, "a", np.mean([at(o, 0, roi) for o in pr["0"]]))
    my, sy = ysy.mean(0), ysy.std(0, ddof=1) / np.sqrt(len(ysy))
    mp, sp = ysp.mean(0), ysp.std(0, ddof=1) / np.sqrt(len(ysp))
    fy = cc.fit(xr, my, "lorentz", 5.0); fp = cc.fit(xp, mp, "gauss", 5.0)
    mean_se = np.mean(sy[np.abs(xr) <= 5])
    xs = np.arange(0, 7.001, 0.01)
    dfit = 100 * cc.lorentz(xs, 1, fy["w"]) - 100 * cc.gauss(xs, 1, fp["w"])
    d_b = xs[np.argmax(dfit > mean_se)] if (dfit > mean_se).any() else np.nan
    sides = []
    for sign in (-1, 1):
        idx = [i for i in np.argsort(np.abs(xr)) if np.sign(xr[i]) == sign]
        hit = [abs(xr[i]) for i in idx if cc.gap_exceeds(my[i], sy[i], mp[i], sp[i]) and my[i] > mp[i]]
        sides.append(min(hit) if hit else np.nan)
    rows.append(dict(roi=roi, side_neg=sides[0], side_pos=sides[1], reading_a=np.nanmean(sides), reading_b=d_b, mean_se_yp=mean_se))
    print(f"ROI reading {roi}: x = {np.round(xr[xr >= 0], 2).tolist()}")
    print("   YP% ", np.round(my[xr >= 0], 1).tolist(), " s.e.", np.round(sy[xr >= 0], 1).tolist())
    print("   Pr% ", np.round(mp[xr >= 0], 1).tolist(), " s.e.", np.round(sp[xr >= 0], 1).tolist())
R5r = pd.DataFrame(rows); print(R5r.round(3).to_string())

In [ ]:
# O2 reading (b), the four half-diagonals pooled (one side), and the August YP wells (n = 7) as the
# YP group, for both ROI readings
rows = []
for ypset, ypg in (("all YP wells", yp["0"]), ("August YP wells", aug["0"])):
    for roi in ROIS:
        for pool in POOLS:
            xr, ysy = pct_profile(ypg, roi, pool, np.mean([at(o, 0, roi) for o in ypg]))
            xp, ysp = pct_profile(pr["0"], roi, pool, np.mean([at(o, 0, roi) for o in pr["0"]]))
            my, sy = ysy.mean(0), ysy.std(0, ddof=1) / np.sqrt(len(ysy))
            mp, sp_ = ysp.mean(0), ysp.std(0, ddof=1) / np.sqrt(len(ysp))
            fy = cc.fit(xr, my, "lorentz", 5.0); fp = cc.fit(xp, mp, "gauss", 5.0)
            mean_se = np.mean(sy[np.abs(xr) <= 5])
            xs = np.arange(0, 7.001, 0.01)
            dfit = 100 * cc.lorentz(xs, 1, fy["w"]) - 100 * cc.gauss(xs, 1, fp["w"])
            d_b = xs[np.argmax(dfit > mean_se)] if (dfit > mean_se).any() else np.nan
            sides = []
            for sign in ((1,) if pool == "b" else (-1, 1)):
                idx = [i for i in np.argsort(np.abs(xr)) if np.sign(xr[i]) == sign]
                hit = [abs(xr[i]) for i in idx if cc.gap_exceeds(my[i], sy[i], mp[i], sp_[i]) and my[i] > mp[i]]
                sides.append(min(hit) if hit else np.nan)
            rows.append(dict(yp_set=ypset, roi=roi, pool=pool, reading_a=np.nanmean(sides), reading_b=d_b))
print(pd.DataFrame(rows).round(2).to_string())

In [ ]:
# Footprint readings (O1 i to iv, claims file as amended 2026-11-20), with the darkest footprint as reference
def chk_5rev():
    rows = []
    for roi in ROIS:
        for pool in POOLS:
            xr, ysy = pct_profile(yp["0"], roi, pool, np.mean([at(o, 0, roi) for o in yp["0"]]))
            xp, ysp = pct_profile(pr["0"], roi, pool, np.mean([at(o, 0, roi) for o in pr["0"]]))
            my, sy = ysy.mean(0), ysy.std(0, ddof=1) / np.sqrt(len(ysy)); mp, sp_ = ysp.mean(0), ysp.std(0, ddof=1) / np.sqrt(len(ysp))
            fy = cc.fit(xr, my, "lorentz", 5.0); fp = cc.fit(xp, mp, "gauss", 5.0)
            mean_se = np.mean(sy[np.abs(xr) <= 5]); xs = np.arange(0, 7.001, 0.01)
            dfit = 100 * cc.lorentz(xs, 1, fy["w"]) - 100 * cc.gauss(xs, 1, fp["w"])
            sides = []
            for sign in ((1,) if pool == "b" else (-1, 1)):
                idx = [i for i in np.argsort(np.abs(xr)) if np.sign(xr[i]) == sign]
                hit = [abs(xr[i]) for i in idx if cc.gap_exceeds(my[i], sy[i], mp[i], sp_[i]) and my[i] > mp[i]]
                sides.append(min(hit) if hit else np.nan)
            rows.append(dict(roi=roi, pool=pool, reading_a=np.nanmean(sides), reading_b=xs[np.argmax(dfit > mean_se)] if (dfit > mean_se).any() else np.nan))
    return rows
T = by_footprint(chk_5rev); print(T.round(2).to_string())
q = T[T.footprint != "darkest"]
print(f"readings (i) to (iv): reading (b) {q.reading_b.min():.2f} to {q.reading_b.max():.2f} mm, inside 1.8 to 2.2 in {int(q.reading_b.between(1.8, 2.2).sum())} of {len(q)}; reading (a) {q.reading_a.min():.2f} to {q.reading_a.max():.2f} mm")

**Result: reproduced.** Under reading (b), the fitted Lorentzian YP profile exceeds the fitted Gaussian Pr profile by more than the mean s.e. of YP% from 2.08 mm (whole-mm ROIs) and 2.11 mm (contiguous ROIs), inside 1.8 to 2.2 mm. Under reading (a), on the data by the graph rule, the first such distance is 3.0 mm (whole-mm ROIs) and 2.84 mm (contiguous ROIs, the mean of 3.25 and 2.44 mm on the two sides), outside the range; the criterion asks for one reading. With the four half-diagonals pooled (O2 reading b), reading (b) gives 2.06 and 2.09 mm and reading (a) 3.00 and 2.84 mm. With the 7 August YP wells, reading (b) gives 1.78 to 1.83 mm and reading (a) 2.44 to 3.00 mm.

Under the footprint readings (i) to (iv), reading (b) gives 2.04 to 2.15 mm in all 16 combinations of ROI reading and pooling, and reading (a) 2.84 to 3.50 mm.

### fig5c-nuclear-yp-centre

> In the central spot cells showed bright nuclear YP staining, indicating irreversible damage, while farther out reversibly electroporated cells were weakly stained with less contrast between nucleus and cytoplasm; cells inside the electrode footprint were unstained though present (Hoechst).

The high-resolution images of Fig. 5C are not in the dandiset (every 001284 file is a 4x whole-well image), so the nuclear part cannot be tested. Intensity part on the 0-mm whole-well images: background-subtracted YP in zone 1 (within 1 mm) against zone 2 (2 to 4 mm, reading a; 3 to 5 mm, reading b) and zone 3 (the 0.5-mm band outside a wall, 7.15 to 7.65 mm), and footprint YP (mean of the four footprints) below zones 2 and 3. Footprint part: the Hoechst channel of the 2-h images, footprint mean against the camera offset (0.1st percentile of the image, a standard choice) and against the well's median; cells are taken as present when the footprint is at least half-way from the offset to the median (a standard choice).

In [ ]:
rows = []
for o in yp["0"]:
    z = o["zone"]; b = z["background"]
    rows.append(dict(well=o["subject"], zone1=z["zone1"] - b, zone2a=z["zone2a"] - b, zone2b=z["zone2b"] - b, zone3=z["zone3"] - b, footprint=z["footprint"] - b))
Z = pd.DataFrame(rows); print(Z.round(1).to_string())
m = Z.mean(numeric_only=True); s = Z.sem(numeric_only=True)
print("means:", m.round(1).to_dict())
for z2 in ("zone2a", "zone2b"):
    print(f"zone1 > {z2}: {bool(m.zone1 > m[z2])} (gap rule {cc.gap_exceeds(m.zone1, s.zone1, m[z2], s[z2])}); zone1 > zone3: {bool(m.zone1 > m.zone3)}; footprint < {z2}: {bool(m.footprint < m[z2])}; footprint < zone3: {bool(m.footprint < m.zone3)}")
print("wells where every inequality holds (zone2a):", int(((Z.zone1 > Z.zone2a) & (Z.zone1 > Z.zone3) & (Z.footprint < Z.zone2a) & (Z.footprint < Z.zone3)).sum()), "of", len(Z))
rows = []
for o in wells("001284", "DAPI_2h", elevation_mm="0"):
    z = o["zone"]
    rows.append(dict(well=o["subject"], footprint=z["footprint"], offset=z["offset"], median=z["median"], frac=(z["footprint"] - z["offset"]) / (z["median"] - z["offset"])))
H = pd.DataFrame(rows); print("Hoechst 2 h after exposure, 0 mm:"); print(H.round(2).to_string())
print("wells with footprint Hoechst at least half-way from offset to median:", int((H.frac >= 0.5).sum()), "of", len(H))

**Result: partially reproduced.** Intensity part: mean background-subtracted YP is 1607.5 a.u. in zone 1, 140.2 (2 to 4 mm) or 86.1 (3 to 5 mm) in zone 2, 320.8 in zone 3 and 10.5 in the footprints; every inequality holds in 12 of 12 wells. Footprint part: in the 5 wells with a Hoechst image taken 2 h after exposure, the footprint Hoechst signal is above the well's median (1.21 to 1.57 of the way from the camera offset to the median), so cells are present. The nuclear part cannot be tested, as the high-resolution images of Fig. 5C are not in the dandiset.

## Figure 6: pulse duration (DANDI:001277)

The paper reports n = 5 per duration. The dandiset's 600-ns group holds 7 wells. In P1-20240926-B2 the YP spot is 5.4 mm from the array centre; P1-20240926-B3 has its YP maximum at the centre but more uptake 3 to 5 mm out than the other wells of this dandiset (measured under fig6d-longer-pulses-sharper-peak). The main reading uses the five other 600-ns wells, so that n = 5 as the claims state; all seven, and the six without B2, are shown as variants. The sham well is left out.

In [ ]:
f6 = {d: wells("001277", pulse_ns=d, sham=False) for d in ("300", "600", "900")}
odd = {"P1-20240926-B2", "P1-20240926-B3"}
for d, g in f6.items():
    print(d, [(o["subject"], round(at(o, 0)), round(cc.fit(*prof(o), "lorentz", 4.0)["fwhm"], 2)) for o in g])
f6_main = {d: [o for o in g if o["subject"] not in odd] for d, g in f6.items()}
# reading with only B2 left out (its YP spot is 5.4 mm from the array centre), n = 6 at 600 ns
f6_noB2 = {d: [o for o in g if o["subject"] != "P1-20240926-B2"] for d, g in f6.items()}
print("n per duration, main:", {d: len(g) for d, g in f6_main.items()}, "variant:", {d: len(g) for d, g in f6.items()}, "without B2:", {d: len(g) for d, g in f6_noB2.items()})

### fig6b-peak-900ns-not-significant

> The central YP peak was highest for 900-ns pulses, but not significantly different from 300- or 600-ns pulses.

In [ ]:
for label, groups in (("main (n = 5 each)", f6_main), ("variant, all 600-ns wells", f6), ("variant, without B2 only", f6_noB2)):
    c = {d: np.array([at(o, 0) for o in g]) for d, g in groups.items()}
    print(label, {d: (round(v.mean()), round(ms(v)[1])) for d, v in c.items()}, "highest:", max(c, key=lambda d: c[d].mean()))
    for other in ("300", "600"):
        t = stats.ttest_ind(c["900"], c[other])
        dz = (c["900"].mean() - c[other].mean()) / np.sqrt((c["900"].var(ddof=1) + c[other].var(ddof=1)) / 2)
        print(f"   900 vs {other}: t = {t.statistic:.2f}, p = {t.pvalue:.3f}, Bonferroni x2 p = {min(1, 2 * t.pvalue):.3f}, x3 p = {min(1, 3 * t.pvalue):.3f}; Cohen's d = {dz:.2f}")
    print(f"   one-way ANOVA p = {stats.f_oneway(*c.values()).pvalue:.3f}; 300 vs 600 p = {stats.ttest_ind(c['300'], c['600']).pvalue:.3f}")

**Result: reproduced.** Mean centre YP is 538, 800 and 891 a.u. for 300, 600 and 900 ns (n = 5 each), highest at 900 ns; 900 against 300 ns p = 0.288 (Bonferroni 0.576), against 600 ns p = 0.801 (Bonferroni 1.0). ANOVA p = 0.506. With all seven 600-ns wells 900 ns is still highest and no test is significant (ANOVA p = 0.470), and the same holds with only B2 left out (n = 6; ANOVA p = 0.481, 900 against 600 ns p = 0.759). Over the 21 ways of choosing five of the seven 600-ns wells, 900 ns is highest with no corrected p below 0.05 in 20; in the other, 900 ns is not the highest.

### fig6c-normalized-decline-similar

> Normalized to the central peak, the sideways decline of YP was similar for the three pulse durations.

Each experiment in % of its value at x = 0, then mean +/- s.e. per duration; at every distance every pair of durations is checked with the graph rule. Variants: normalising the mean profile, and -5 to 5 mm only.

In [ ]:
for label, groups in (("main", f6_main), ("variant, all 600-ns wells", f6)):
    for roi in ROIS:
        for pool in POOLS:
            prof_pct = {}
            for d, g in groups.items():
                x, ys = mean_profile(g, roi, pool)
                prof_pct[d] = (x, ys / ys[:, [np.argmin(np.abs(x))]] * 100)
            hits = []
            for i, xi in enumerate(x):
                if xi == 0: continue
                for a_, b_ in (("300", "600"), ("300", "900"), ("600", "900")):
                    ya, yb = prof_pct[a_][1][:, i], prof_pct[b_][1][:, i]
                    if cc.gap_exceeds(ya.mean(), ms(ya)[1], yb.mean(), ms(yb)[1]):
                        hits.append((round(float(xi), 2), a_, b_, round(ya.mean(), 1), round(yb.mean(), 1)))
            within5 = [h for h in hits if abs(h[0]) <= 5]
            print(f"{label}, ROI {roi}, pooling {pool}: {len(hits)} distance-pairs with a gap ({len(within5)} within 5 mm): {hits}")
x, _ = mean_profile(f6_main["300"])
for d, g in f6_main.items():
    xx, ys = mean_profile(g); mp = ys.mean(0); print(d, "mean profile normalised to its peak (%):", np.round(mp / mp[np.argmin(np.abs(xx))] * 100, 1)[xx >= 0].tolist())

In [ ]:
# Footprint readings (O1 i to iv, claims file as amended 2026-11-20), with the darkest footprint as reference
def chk_6c():
    rows = []
    for label, groups in (("main", f6_main), ("all 600", f6)):
        for roi in ROIS:
            for pool in POOLS:
                pct = {}
                for d, g in groups.items():
                    x, ys = mean_profile(g, roi, pool); pct[d] = ys / ys[:, [np.argmin(np.abs(x))]] * 100
                hits = [(xi, a_, b_) for i, xi in enumerate(x) if xi != 0 for a_, b_ in (("300", "600"), ("300", "900"), ("600", "900"))
                        if cc.gap_exceeds(pct[a_][:, i].mean(), ms(pct[a_][:, i])[1], pct[b_][:, i].mean(), ms(pct[b_][:, i])[1])]
                rows.append(dict(set=label, roi=roi, pool=pool, n_gaps=len(hits), pairs=sorted({f"{a_}-{b_}" for _, a_, b_ in hits})))
    return rows
T = by_footprint(chk_6c); print(T.to_string())
q = T[T.footprint != "darkest"]
print("readings (i) to (iv), main set: gaps under every reading", bool((q[q.set == "main"].n_gaps > 0).all()), f"({q[q.set == 'main'].n_gaps.min()} to {q[q.set == 'main'].n_gaps.max()}); all 600-ns wells: {q[q.set == 'all 600'].n_gaps.min()} to {q[q.set == 'all 600'].n_gaps.max()}")

**Result: not reproduced.** By the paper's graph rule, durations differ by a gap larger than the error bars at some distances under every reading. With n = 5 per duration there are 5 to 20 such distance and pair combinations, nearly all within 5 mm, mostly 300 ns above 600 or 900 ns at 1 to 3 mm (whole-mm ROIs at 2 mm: 20.2% against 12.3% of the peak for 300 and 600 ns). With all seven 600-ns wells there are 2 to 10, mostly 300 against 900 ns; under the contiguous-ROI readings 600 against 900 ns also differ at +/-0.81 mm (74.9% and 78.3% against 44.4% and 54.0% of the peak). The 300-ns peak is broader, which is what fig6d-longer-pulses-sharper-peak measures.

Under the footprint readings (i) to (iv) durations differ by a gap at some distance under every reading: 4 to 26 distance and pair combinations with n = 5 per duration, 2 to 16 with all seven 600-ns wells.

### fig6d-longer-pulses-sharper-peak

> Fitted per experiment, the FWHM of the central YP peak decreases with pulse duration from 300 to 900 ns, with a trend slope different from zero (p = 0.003).

Per-experiment Lorentzian FWHM (a and b free, centre fixed, no offset), over the fit ranges of O3 (4, 5, 6 mm) and the ROI readings of O1; O2 does not change a fit to a symmetric function of x. Slope of FWHM against duration and against log duration, on individual experiments (O4 a) and on group means (O4 b).

In [ ]:
rows = []
for label, groups in (("main", f6_main), ("all 600", f6), ("no B2", f6_noB2)):
    for roi in ROIS:
        for hr in RANGES:
            fw = {d: fwhm_each(g, roi, hr) for d, g in groups.items()}
            X = np.concatenate([[int(d)] * len(v) for d, v in fw.items()]); Y = np.concatenate(list(fw.values()))
            Xm = np.array([int(d) for d in fw]); Ym = np.array([v.mean() for v in fw.values()])
            r = dict(set=label, roi=roi, range=hr, **{f"fwhm_{d}": v.mean() for d, v in fw.items()}, **{f"se_{d}": ms(v)[1] for d, v in fw.items()})
            for nm, (xx, yy) in (("lin_exp", (X, Y)), ("log_exp", (np.log(X), Y)), ("lin_means", (Xm, Ym)), ("log_means", (np.log(Xm), Ym))):
                st = cc.slope_test(xx, yy); r[f"{nm}_slope"] = st["slope"]; r[f"{nm}_p"] = st["p"]
            rows.append(r)
R6d = pd.DataFrame(rows); print(R6d.round(4).to_string())
for label in ("main", "no B2"):
    mm = R6d[R6d.set == label]
    print(f"{label}: every slope negative:", bool((mm[[c for c in mm if c.endswith('_slope')]] < 0).all().all()),
          "; p on experiments", f"{mm[['lin_exp_p', 'log_exp_p']].min().min():.4f} to {mm[['lin_exp_p', 'log_exp_p']].max().max():.4f}",
          "; p on means", f"{mm[['lin_means_p', 'log_means_p']].min().min():.3f} to {mm[['lin_means_p', 'log_means_p']].max().max():.3f}")

Every way of choosing 5 of the 7 600-ns wells (21 subsets), for fig6d-longer-pulses-sharper-peak, fig6d-600-900-differ-from-300 and fig6b-peak-900ns-not-significant. Plates P2 and P3 each hold two wells of each duration; plate P1 holds one 300-ns well, one 900-ns well, a sham and three 600-ns wells (A3, B2, B3), so n = 5 per duration is reached by leaving out two of P1's 600-ns wells.

In [ ]:
# every 5-of-7 subset of the 600-ns wells (21 subsets), for the claims of Figure 6
from itertools import combinations
FW6 = {(roi, hr): {d: fwhm_each(g, roi, hr) for d, g in f6.items()} for roi in ROIS for hr in RANGES}
C6 = {d: np.array([at(o, 0) for o in g]) for d, g in f6.items()}
names600 = [o["subject"] for o in f6["600"]]
rows = []
for keep in combinations(range(len(f6["600"])), 5):
    keep = list(keep); left = ", ".join(n[3:] for i, n in enumerate(names600) if i not in keep)
    c600 = C6["600"][keep]
    r = dict(left_out=left, b6_900_highest=bool(C6["900"].mean() > max(C6["300"].mean(), c600.mean())),
             b6_min_p_x2=min(1, 2 * min(stats.ttest_ind(C6["900"], C6["300"]).pvalue, stats.ttest_ind(C6["900"], c600).pvalue)))
    p_exp, p_means, neg, p600, p900 = [], [], [], [], []
    for (roi, hr), fw in FW6.items():
        f = dict(fw); f["600"] = fw["600"][keep]
        X = np.concatenate([[int(d)] * len(v) for d, v in f.items()]); Y = np.concatenate(list(f.values()))
        Xm = np.array([int(d) for d in f]); Ym = np.array([v.mean() for v in f.values()])
        for xx, yy, store in ((X, Y, p_exp), (np.log(X), Y, p_exp), (Xm, Ym, p_means), (np.log(Xm), Ym, p_means)):
            s = cc.slope_test(xx, yy); store.append(s["p"]); neg.append(s["slope"] < 0)
        p600.append(min(1, 2 * stats.ttest_ind(f["600"], f["300"]).pvalue) if f["600"].mean() < f["300"].mean() else 1.0)
        p900.append(min(1, 2 * stats.ttest_ind(f["900"], f["300"]).pvalue))
    r.update(all_slopes_negative=all(neg), p_exp_max=max(p_exp), p_means_max=max(p_means), p600_x2_min=min(p600), p600_x2_max=max(p600), p900_x2_max=max(p900))
    rows.append(r)
S6 = pd.DataFrame(rows); print(S6.round(4).to_string())
print(f"subsets with fig6d-longer reproduced on experiments (every slope negative, p < 0.05 under every reading of O1 and O3): {int(((S6.p_exp_max < 0.05) & S6.all_slopes_negative).sum())} of {len(S6)}")
print(f"subsets with the 600-ns comparison below 0.05 (x2) under every reading: {int((S6.p600_x2_max < 0.05).sum())}; under at least one: {int((S6.p600_x2_min < 0.05).sum())}")
print(f"subsets where 900 ns is highest with no corrected p below 0.05 (fig6b): {int((S6.b6_900_highest & (S6.b6_min_p_x2 >= 0.05)).sum())} of {len(S6)}")

Is the shape of P1-20240926-B2 and B3 different from the other wells by a measure other than the FWHM (verifier, round 2)? For every 0.2-MHz CANCAN YP well (001276, 001277, and 001284 at 0 mm): the distance of the YP spot centre from the array centre (wells located from their footprints), YP at 1, 2, 3 and 5 mm as a fraction of the centre (four half-diagonals averaged, whole-mm ROIs), the ratio of the centre to the mean of 3 to 5 mm, and the distance along the diagonals (contiguous ROIs) at which the profile is highest. None of these is the FWHM, but YP away from the centre relative to the centre measures the same broadness the FWHM summarises.

In [ ]:
rows = []
for o in W:
    if o["chan"] != "FITC" or o["protocol"] != "CanCan" or o["packet_rate"] != "0.2MHz" or o["sham"]: continue
    if o["dandiset"] == "001284" and o["elevation_mm"] != "0": continue
    h = o["pa"]["half"].mean(0); hb = o["pb"]["half"].mean(0)
    off_ = GEO[(GEO.well == o["subject"]) & (GEO.dandiset == o["dandiset"])].spot_offset_mm.iloc[0] if o["how"] == "footprints" else np.nan
    rows.append(dict(dandiset=o["dandiset"], group=group_of(o), well=o["subject"], spot_offset_mm=off_, c0=h[0],
                     **{f"y{k}_frac": h[k] / h[0] for k in (1, 2, 3, 5)}, centre_over_3to5=h[0] / h[3:6].mean(),
                     max_at_mm=float(o["pb"]["dists"][np.argmax(hb)])))
PK = pd.DataFrame(rows)
oddw = PK.well.isin(odd) & (PK.dandiset == "001277")
print("P1-20240926-B2 and B3:"); print(PK[oddw].round(3).to_string())
print("\nthe other 0.2-MHz CANCAN wells, by group (min to max):")
cols = ["spot_offset_mm", "y1_frac", "y2_frac", "y3_frac", "y5_frac", "centre_over_3to5", "max_at_mm"]
print(PK[~oddw].groupby(["dandiset", "group"])[cols].agg(["min", "max"]).round(2).to_string())
print("\nall other wells:", {c: (round(PK[~oddw][c].min(), 2), round(PK[~oddw][c].max(), 2)) for c in cols})
for c in ("y3_frac", "y5_frac", "centre_over_3to5"):
    print(f"{c}: other wells beyond B3's value ({PK[oddw & (PK.well == 'P1-20240926-B3')][c].iloc[0]:.3f}):",
          PK[~oddw & ((PK[c] >= PK[oddw & (PK.well == 'P1-20240926-B3')][c].iloc[0]) if c != "centre_over_3to5" else (PK[c] <= PK[oddw & (PK.well == 'P1-20240926-B3')][c].iloc[0]))][["group", "well", c]].round(3).to_dict("records"))


In [ ]:
# Footprint readings (O1 i to iv, claims file as amended 2026-11-20), with the darkest footprint as reference
def chk_6d():
    rows = []
    for label, groups in (("main", f6_main), ("no B2", f6_noB2), ("all 600", f6)):
        for roi in ROIS:
            for hr in RANGES:
                fw = {d: fwhm_each(g, roi, hr) for d, g in groups.items()}
                X = np.concatenate([[int(d)] * len(v) for d, v in fw.items()]); Y = np.concatenate(list(fw.values()))
                Xm = np.array([int(d) for d in fw]); Ym = np.array([v.mean() for v in fw.values()])
                r = dict(set=label, roi=roi, range=hr, **{f"fwhm_{d}": v.mean() for d, v in fw.items()})
                for nm, (xx, yy) in (("lin_exp", (X, Y)), ("log_exp", (np.log(X), Y)), ("lin_means", (Xm, Ym)), ("log_means", (np.log(Xm), Ym))):
                    st = cc.slope_test(xx, yy); r[f"{nm}_slope"] = st["slope"]; r[f"{nm}_p"] = st["p"]
                rows.append(r)
    return rows
T6d = by_footprint(chk_6d); print(T6d.round(4).to_string())
for label in ("main", "no B2", "all 600"):
    q = T6d[(T6d.set == label) & (T6d.footprint != "darkest")]
    neg = bool((q[["lin_exp_slope", "log_exp_slope"]] < 0).all().all())
    print(f"{label}, readings (i) to (iv): slopes on experiments negative {neg}, p {q[['lin_exp_p', 'log_exp_p']].min().min():.4f} to {q[['lin_exp_p', 'log_exp_p']].max().max():.4f}; on means p {q[['lin_means_p', 'log_means_p']].min().min():.3f} to {q[['lin_means_p', 'log_means_p']].max().max():.3f}")

**Result: not reproduced** (changed from reproduced, resting on assumptions, after the third verification round). With only P1-20240926-B2 left out (n = 6 at 600 ns), the well choice the data support, the slopes are negative with p = 0.51 to 0.77 on experiments and 0.63 to 0.83 on group means, and the 600-ns mean FWHM is above the 300-ns mean; with all seven 600-ns wells p = 0.86 to 0.99. With B2 and B3 left out, per-experiment FWHM as points (O4 a) and n = 5 per duration, the slope is negative under every ROI reading and fit range, p = 0.0008 to 0.0058 against duration and against log duration (paper p = 0.003). On the three group means (O4 b) the slopes are negative with p = 0.048 to 0.165. With all seven 600-ns wells no slope is significant (p = 0.86 to 0.99, and the slopes against log duration are positive). Over the 21 ways of choosing five of the seven 600-ns wells, the criterion holds only for the main choice, which leaves out B2 and B3; in the other 20 the largest p is 0.79 to 0.98.

The outcome recorded after round 1, reproduced, rested on two assumptions. (1) The regression points: with three group means the slope test has one degree of freedom, and p = 0.003 would need t of about 212, so per-experiment points are the better-supported reading. (2) The 600-ns group. B2 has a reason to be left out that does not depend on its FWHM: its YP spot is 5.39 mm from the array centre, where every other 0.2-MHz CANCAN well with footprints has it within 1.21 mm. B3 does not. Its spot is 0.33 mm from the centre, its profile is highest at the centre (the earlier statement that it has no central peak was wrong), and its centre value (748 a.u.) is inside the range of the other 600-ns wells. What sets it apart is the uptake away from the centre: YP at 3 and 5 mm is 46% and 35% of the centre, against 5 to 13% and 3 to 9% in every other well of 001277. That is the broadness the FWHM measures, the quantity under test, and the 001276 CT26 wells at 6 bursts reach 41 to 66% at 3 mm, so the shape is not outside what these exposures produce. Leaving B3 out is therefore not independent of the result. The choice the data support is to leave out B2 alone, and with it, as with all seven wells, the slope is not significant. The criterion needs p < 0.05 under every reading, so the claim is not reproduced, on per-experiment points and on group means.

Under the footprint readings (i) to (iv) of the amended O1, with the main wells every slope on experiments is negative with p = 0.0003 to 0.0465 (on group means p = 0.000 to 0.154); with only B2 left out p = 0.45 to 0.88, and with all seven 600-ns wells p = 0.84 to 0.98 with some slopes positive. The footprint readings leave the outcome as it is.

### fig6d-600-900-differ-from-300

> FWHM for 600-ns and for 900-ns packets is each significantly smaller than for 300-ns packets (Bonferroni-corrected t-tests, p = 0.028 for one and p = 0.0072 for the other).

In [ ]:
rows = []
for label, groups in (("main", f6_main), ("all 600", f6), ("no B2", f6_noB2)):
    for roi in ROIS:
        for hr in RANGES:
            fw = {d: fwhm_each(g, roi, hr) for d, g in groups.items()}
            for d in ("600", "900"):
                t = stats.ttest_ind(fw[d], fw["300"])
                rows.append(dict(set=label, roi=roi, range=hr, vs300=d, diff=fw[d].mean() - fw["300"].mean(), p=t.pvalue, p_x2=min(1, 2 * t.pvalue), p_x3=min(1, 3 * t.pvalue)))
R6p = pd.DataFrame(rows); print(R6p.round(4).to_string())
for label in ("main", "no B2"):
  mm = R6p[R6p.set == label]
  for d in ("600", "900"):
    q = mm[mm.vs300 == d]
    print(f"{label}, {d} vs 300: smaller FWHM in every reading {bool((q['diff'] < 0).all())}; corrected p (x2) {q.p_x2.min():.4f} to {q.p_x2.max():.4f}; (x3) {q.p_x3.min():.4f} to {q.p_x3.max():.4f}")

In [ ]:
# Footprint readings (O1 i to iv, claims file as amended 2026-11-20), with the darkest footprint as reference
def chk_6p():
    rows = []
    for label, groups in (("main", f6_main), ("no B2", f6_noB2)):
        for roi in ROIS:
            for hr in RANGES:
                fw = {d: fwhm_each(g, roi, hr) for d, g in groups.items()}
                for d in ("600", "900"):
                    p = stats.ttest_ind(fw[d], fw["300"]).pvalue
                    rows.append(dict(set=label, roi=roi, range=hr, vs300=d, diff=fw[d].mean() - fw["300"].mean(), p_x2=min(1, 2 * p), p_x3=min(1, 3 * p)))
    return rows
T = by_footprint(chk_6p); print(T.round(4).to_string())
# the criterion per reading: both comparisons below 0.05 (reproduced), one (partial), none
W6 = T[T.set == "main"].pivot_table(index=["footprint", "roi", "range"], columns="vs300", values=["p_x2", "diff"])
W6[("n_below", "")] = ((W6[("p_x2", "600")] < 0.05) & (W6[("diff", "600")] < 0)).astype(int) + ((W6[("p_x2", "900")] < 0.05) & (W6[("diff", "900")] < 0)).astype(int)
print("main set, comparisons (x2) below 0.05 per reading, by footprint:", {fp: g.n_below.tolist() for fp, g in W6.groupby(level=0)})
for label in ("main", "no B2"):
    for d in ("600", "900"):
        q = T[(T.set == label) & (T.vs300 == d) & (T.footprint != "darkest")]
        print(f"{label}, {d} vs 300, readings (i) to (iv): smaller in every reading {bool((q['diff'] < 0).all())}; corrected p (x2) {q.p_x2.min():.4f} to {q.p_x2.max():.4f}, below 0.05 in {int(((q.p_x2 < 0.05) & (q['diff'] < 0)).sum())} of {len(q)}; (x3) {q.p_x3.min():.4f} to {q.p_x3.max():.4f}")

**Result: blocked (methods_underspecified)** (changed from partially reproduced after the third verification round), as the outcome depends on the background footprint, shown below. With the darkest footprint of each well, the choice made before the reader's answer: 900 against 300 ns: FWHM smaller by 0.36 to 0.40 mm, corrected p 0.0063 to 0.0118 (two comparisons) and 0.0095 to 0.0178 (three) under every reading, near the paper's 0.0072. 600 against 300 ns: smaller by 0.25 to 0.28 mm, corrected p 0.049 to 0.169 (two) and 0.074 to 0.253 (three), below 0.05 under one reading only. Over the 21 ways of choosing five of the seven 600-ns wells, the 600-ns comparison is below 0.05 (two comparisons) under at least one reading only in the main choice, and under every reading in none; the 900-ns comparison does not depend on the choice (corrected p at most 0.0118). With only B2 left out, the 600-ns FWHM is larger than at 300 ns under every reading and the 900-ns comparison is unchanged, so the outcome is partial under that choice as well.

Under the footprint readings (i) to (iv) of the amended O1, the 900-ns comparison is below 0.05 (two comparisons) in 11 of 24 combinations (corrected p 0.0021 to 0.1126) and the 600-ns comparison in 3 of 24 (0.031 to 0.78). Counting the comparisons below 0.05 per reading of ROI and fit range, footprint (i) gives none in all six, (ii) one in two and none in four, (iii) one in three and none in three, and (iv) one in three and both in three. The criterion's outcome therefore depends on the footprint: not reproduced with (i), partially reproduced or not with (ii) and (iii), reproduced in some readings with (iv). The paper does not say which footprint it used, and no position is better supported than another. The claim is blocked (methods_underspecified). The block also stands with only B2 left out, where the 600-ns FWHM is larger than at 300 ns.

### fig6d-900ns-fwhm

> Packets of 900-ns pulses reduced FWHM to 1.7 +/- 0.08 mm, 11.1 +/- 0.6% of the gap between electrodes.

In [ ]:
for roi in ROIS:
    for hr in RANGES:
        v = fwhm_each(f6["900"], roi, hr)
        m, se, n = ms(v)
        print(f"ROI {roi}, range {hr}: 900-ns FWHM {m:.3f} +/- {se:.3f} mm (n={n}), {100 * m / cc.DIAG_GAP_MM:.2f} +/- {100 * se / cc.DIAG_GAP_MM:.2f}% of 15.3 mm; within 1.7 +/- 0.08: {abs(m - 1.7) <= 0.08}; within 11.1 +/- 0.6: {abs(100 * m / 15.3 - 11.1) <= 0.6}")

In [ ]:
# Sensitivity to locating the array (problem B4 of round 1): 3 of the 5 wells were measured from the
# YP spot. Shown: the 2 footprint wells alone, and the spot wells shifted by the median paired
# FWHM difference (spot minus footprints) of the CANCAN wells where both are known.
for roi in ROIS:
    for hr in RANGES:
        v = fwhm_each(f6["900"], roi, hr); sp = np.array([o["how"] == "spot centre" for o in f6["900"]])
        adj = np.where(sp, v - FWHM_BIAS[(roi, hr)], v)
        print(f"ROI {roi}, range {hr}: footprint wells only {v[~sp].mean():.3f} mm (n={int((~sp).sum())}); spot wells adjusted by {-FWHM_BIAS[(roi, hr)]:+.3f}: {adj.mean():.3f} mm, {100 * adj.mean() / 15.3:.2f}%; within both tolerances: {abs(adj.mean() - 1.7) <= 0.08 and abs(100 * adj.mean() / 15.3 - 11.1) <= 0.6}")

In [ ]:
# Footprint readings (O1 i to iv, claims file as amended 2026-11-20), with the darkest footprint as reference
def chk_6f():
    rows = []
    for roi in ROIS:
        for hr in RANGES:
            m, se, n = ms(fwhm_each(f6["900"], roi, hr))
            rows.append(dict(roi=roi, range=hr, fwhm=m, se=se, pct=100 * m / cc.DIAG_GAP_MM, inside=bool(abs(m - 1.7) <= 0.08 and abs(100 * m / cc.DIAG_GAP_MM - 11.1) <= 0.6)))
    return rows
T = by_footprint(chk_6f); print(T.round(3).to_string())
q = T[T.footprint != "darkest"]
print(f"readings (i) to (iv): FWHM {q.fwhm.min():.3f} to {q.fwhm.max():.3f} mm, {q.pct.min():.2f} to {q.pct.max():.2f}%; both within tolerance in {int(q.inside.sum())} of {len(q)}")

**Result: reproduced.** 900-ns FWHM is 1.688 to 1.767 mm (s.e. 0.049 to 0.052, n = 5), 11.03 to 11.55% of 15.3 mm, inside 1.7 +/- 0.08 mm and 11.1 +/- 0.6% under every reading. Three of the five wells were located from the YP spot, which narrows the FWHM by a median of 0.10 to 0.13 mm in the wells where both ways could be compared. With the two footprint wells alone the FWHM is 1.755 to 1.763 mm under contiguous ROIs and 1.850 to 1.857 mm under whole-mm ROIs; with the spot wells shifted by the median difference it is 1.745 to 1.771 mm (11.41 to 11.57%) under contiguous ROIs, inside both tolerances, and 1.831 to 1.837 mm under whole-mm ROIs, outside. The claim needs one combination, which contiguous ROIs give in each case.

Under the footprint readings (i) to (iv) the 900-ns FWHM is 1.602 to 1.721 mm (10.47 to 11.25%), both values within tolerance in 21 of 24 combinations; the criterion needs one.

## Figure 7: number of bursts (DANDI:001276)

0.2-MHz CANCAN wells by cell line and number of bursts (x4). The dandiset holds more CHO wells than the paper's n = 4 (printed in the data check); all are used, and the per-date breakdown is shown.

In [ ]:
B = {}
for cell in ("CT26", "CHO"):
    for b in ("1", "2", "4", "6"):
        B[(cell, b)] = wells("001276", cell=cell, repeats=b, packet_rate="0.2MHz", protocol="CanCan")
print({k: len(v) for k, v in B.items()})
print({k: sorted({o["subject"][3:11] for o in v}) for k, v in B.items()})

### fig7be-center-linear-with-bursts

> The central YP peak grew linearly with the number of bursts in both CT26 and CHO cells (slopes different from zero at p < 0.001), except that it saturated after 6 bursts (x4) in CT26 cells.

In [ ]:
F7 = {}
for cell in ("CT26", "CHO"):
    bs = ("1", "2", "4") if cell == "CT26" else ("1", "2", "4", "6")
    X = np.concatenate([[int(b)] * len(B[(cell, b)]) for b in bs]); Y = np.concatenate([[at(o, 0) for o in B[(cell, b)]] for b in bs])
    st = cc.slope_test(X, Y); F7[cell] = (X, Y, st)
    Xm = np.array([int(b) for b in bs]); Ym = np.array([np.mean([at(o, 0) for o in B[(cell, b)]]) for b in bs])
    sm = cc.slope_test(Xm, Ym)
    print(f"{cell}: means by bursts {dict(zip(bs, np.round(Ym)))}; experiments n={len(X)}: slope {st['slope']:.1f} a.u./burst, r = {st['r']:.3f}, p = {st['p']:.2e}; group means: slope {sm['slope']:.1f}, p = {sm['p']:.4f}")
    if cell == "CT26":
        X6 = np.concatenate([X, [6] * len(B[("CT26", "6")])]); Y6 = np.concatenate([Y, [at(o, 0) for o in B[("CT26", "6")]]])
        s6 = cc.slope_test(X6, Y6); print(f"   CT26 with 6 bursts kept (variant): slope {s6['slope']:.1f}, p = {s6['p']:.2e}")
    # per date
    for date in sorted({o["subject"][3:11] for b in bs for o in B[(cell, b)]}):
        xx = [int(b) for b in bs for o in B[(cell, b)] if o["subject"][3:11] == date]
        yy = [at(o, 0) for b in bs for o in B[(cell, b)] if o["subject"][3:11] == date]
        print(f"   {date}: {len(xx)} wells, bursts {sorted(set(xx))}")

In [ ]:
# random subsets of the wells matching the paper's n (CT26 3 to 5 per burst number, CHO 4), for
# fig7be-center-linear-with-bursts and fig7cf-fwhm-with-bursts; 2000 draws, a standard number
rng = np.random.default_rng(7)
C7 = {k: np.array([at(o, 0) for o in v]) for k, v in B.items()}
F7w = {(roi, hr): {k: fwhm_each(v, roi, hr) for k, v in B.items()} for roi in ROIS for hr in RANGES}
def draw(k):
    n = len(B[k])
    if k[0] == "CT26":
        size = min(n, rng.integers(3, 6))
    else:
        size = min(n, 4)
    return np.sort(rng.choice(n, size, replace=False))
rows = []
for _ in range(2000):
    idx = {k: draw(k) for k in B}
    r = {}
    for cell, bs in (("CT26", ("1", "2", "4")), ("CHO", ("1", "2", "4", "6"))):
        X = np.concatenate([[int(b)] * len(idx[(cell, b)]) for b in bs]); Y = np.concatenate([C7[(cell, b)][idx[(cell, b)]] for b in bs])
        r[f"be_{cell}_exp"] = cc.slope_test(X, Y)["p"]
        r[f"be_{cell}_means"] = cc.slope_test([int(b) for b in bs], [C7[(cell, b)][idx[(cell, b)]].mean() for b in bs])["p"]
    for cell, bs in (("CT26", ("1", "2", "4", "6")), ("CHO", ("2", "4", "6"))):
        pe, pm = [], []
        for key, fw in F7w.items():
            X = np.concatenate([[int(b)] * len(idx[(cell, b)]) for b in bs]); Y = np.concatenate([fw[(cell, b)][idx[(cell, b)]] for b in bs])
            ok = np.isfinite(Y) & (Y < 50)
            pe.append(cc.slope_test(X[ok], Y[ok])["p"]); pm.append(cc.slope_test([int(b) for b in bs], [np.nanmean(fw[(cell, b)][idx[(cell, b)]]) for b in bs])["p"])
        r[f"cf_{cell}_exp_max"] = max(pe); r[f"cf_{cell}_exp_min"] = min(pe); r[f"cf_{cell}_means_max"] = max(pm); r[f"cf_{cell}_means_min"] = min(pm)
    rows.append(r)
S7 = pd.DataFrame(rows)
print(S7.describe(percentiles=[0.05, 0.5, 0.95]).T.round(4).to_string())
be = (S7.be_CT26_exp < 0.001) & (S7.be_CHO_exp < 0.001)
print(f"fig7be, on experiments: both slopes p < 0.001 in {be.mean():.1%} of draws (CT26 alone {(S7.be_CT26_exp < 0.001).mean():.1%}); on group means: {((S7.be_CT26_means < 0.001) & (S7.be_CHO_means < 0.001)).mean():.1%} (CT26 alone {(S7.be_CT26_means < 0.001).mean():.1%}, CHO alone {(S7.be_CHO_means < 0.001).mean():.1%})")
cf = (S7.cf_CT26_exp_max < 0.001) & (S7.cf_CHO_exp_min >= 0.05)
print(f"fig7cf, on experiments under every reading of O1 and O3: CT26 p < 0.001 and CHO p >= 0.05 in {cf.mean():.1%} of draws (CT26 part {(S7.cf_CT26_exp_max < 0.001).mean():.1%}, CHO part {(S7.cf_CHO_exp_min >= 0.05).mean():.1%}); on group means {((S7.cf_CT26_means_max < 0.001) & (S7.cf_CHO_means_min >= 0.05)).mean():.1%}")

In [ ]:
# Footprint readings (O1 i to iv, claims file as amended 2026-11-20), with the darkest footprint as reference
def chk_7be():
    rows = []
    for cell in ("CT26", "CHO"):
        bs = ("1", "2", "4") if cell == "CT26" else ("1", "2", "4", "6")
        X = np.concatenate([[int(b)] * len(B[(cell, b)]) for b in bs]); Y = np.concatenate([[at(o, 0) for o in B[(cell, b)]] for b in bs])
        st = cc.slope_test(X, Y); sm = cc.slope_test([int(b) for b in bs], [np.mean([at(o, 0) for o in B[(cell, b)]]) for b in bs])
        rows.append(dict(cell=cell, n=len(X), slope=st["slope"], p_exp=st["p"], p_means=sm["p"]))
    return rows
T = by_footprint(chk_7be); print(T.to_string(float_format=lambda v: f"{v:.3g}"))
q = T[T.footprint != "darkest"]
print("readings (i) to (iv), on experiments:", {c: (f"{g.p_exp.min():.3e}", f"{g.p_exp.max():.3e}") for c, g in q.groupby("cell")}, "; on group means:", {c: (round(g.p_means.min(), 4), round(g.p_means.max(), 4)) for c, g in q.groupby("cell")})

**Result: partially reproduced.** On individual experiments (O4 a), centre YP grows with bursts in CT26 (1 to 4 bursts, n = 12, slope 903 a.u. per burst, p = 9.1e-04) and CHO (1 to 6 bursts, n = 28, slope 606, p = 2.4e-11). On group means (O4 b) the CT26 slope has p = 0.036 and the CHO slope p = 0.0015. The dandiset holds 6 CT26 wells at 2 bursts (paper n = 3 to 5) and 5 to 8 CHO wells per burst number (paper n = 4). In 2000 random subsets of the paper's size, the CT26 slope on experiments has p below 0.001 in 38.4% of draws (median 0.0016, largest 0.0055) and the CHO slope in every draw; on group means the CT26 slope has p below 0.001 in no draw and the CHO slope in 12.4% of draws. Both slopes have p below 0.01 on experiments in every draw, but the CT26 p below 0.001 depends on which wells are used, so the claim holds in part. The outcome is partial under both O4 readings: on group means the CT26 p is 0.036, above 0.001, so it does not rest on the choice of regression points.

Under the footprint readings (i) to (iv) of the amended O1, on experiments the CT26 slope has p = 1.004e-03 to 1.808e-03, just above 0.001 under every footprint, and the CHO slope 6.7e-13 to 2.5e-11; on group means CT26 p = 0.030 to 0.055 and CHO p = 0.0005 to 0.0012. With footprint (i) on group means the CT26 slope has p above 0.05 and the claim holds in CHO only, which the criterion also calls partial. The outcome is partially reproduced under every footprint, as before.

### fig7b-ct26-saturation

> In CT26 cells the central YP uptake saturated at 6 bursts (x4), falling below the linear increase seen at fewer bursts, as expected when all central cells are already killed.

In [ ]:
def band_at(X, Y, x0, level=0.95):
    X = np.asarray(X, float); Y = np.asarray(Y, float); n = len(X)
    st = cc.slope_test(X, Y); pred = st["intercept"] + st["slope"] * x0
    res = Y - (st["intercept"] + st["slope"] * X); s = np.sqrt(np.sum(res ** 2) / (n - 2))
    se = s * np.sqrt(1 / n + (x0 - X.mean()) ** 2 / np.sum((X - X.mean()) ** 2))
    q = stats.t.ppf(0.5 + level / 2, n - 2)
    return pred, pred - q * se, pred + q * se
c6 = np.array([at(o, 0) for o in B[("CT26", "6")]]); c4 = np.array([at(o, 0) for o in B[("CT26", "4")]])
X, Y, _ = F7["CT26"]
pred, lo, hi = band_at(X, Y, 6)
print(f"O4 (a), experiments: line at 6 bursts {pred:.0f}, 95% band {lo:.0f} to {hi:.0f}; 6-burst mean {c6.mean():.0f} +/- {ms(c6)[1]:.0f} (n={len(c6)}); below the band: {c6.mean() < lo}")
Xm = np.array([1, 2, 4]); Ym = np.array([np.mean([at(o, 0) for o in B[("CT26", b)]]) for b in ("1", "2", "4")])
pm, lom, him = band_at(Xm, Ym, 6)
print(f"O4 (b), group means: line at 6 {pm:.0f}, 95% band {lom:.0f} to {him:.0f}; below the band: {c6.mean() < lom}")
print(f"reading (b): 6-burst {c6.mean():.0f} +/- {ms(c6)[1]:.0f} against 4-burst {c4.mean():.0f} +/- {ms(c4)[1]:.0f}; 6 above 4 by a gap larger than the error bars: {c6.mean() > c4.mean() and cc.gap_exceeds(c6.mean(), ms(c6)[1], c4.mean(), ms(c4)[1])}")
Xc, Yc, _ = F7["CHO"]
Xc5 = Xc[Xc < 6]; Yc5 = Yc[Xc < 6]; pc, loc, hic = band_at(Xc5, Yc5, 6)
cc6 = Yc[Xc == 6]
print(f"CHO (reported): line through 1-4 bursts at 6: {pc:.0f} ({loc:.0f} to {hic:.0f}); 6-burst mean {cc6.mean():.0f} +/- {ms(cc6)[1]:.0f}")

In [ ]:
# Footprint readings (O1 i to iv, claims file as amended 2026-11-20), with the darkest footprint as reference
def chk_7b():
    X = np.concatenate([[b] * len(B[("CT26", str(b))]) for b in (1, 2, 4)]); Y = np.array([at(o, 0) for b in ("1", "2", "4") for o in B[("CT26", b)]])
    c6 = np.array([at(o, 0) for o in B[("CT26", "6")]]); c4 = np.array([at(o, 0) for o in B[("CT26", "4")]])
    pred, lo, hi = band_at(X, Y, 6)
    Ym = np.array([np.mean([at(o, 0) for o in B[("CT26", b)]]) for b in ("1", "2", "4")]); pm, lom, him = band_at(np.array([1, 2, 4]), Ym, 6)
    return [dict(mean6=c6.mean(), se6=ms(c6)[1], line6=pred, band_lo_exp=lo, band_lo_means=lom, below_band_exp=bool(c6.mean() < lo), below_band_means=bool(c6.mean() < lom),
                 below_line=bool(c6.mean() < pred), mean4=c4.mean(), not_above_4=not (c6.mean() > c4.mean() and cc.gap_exceeds(c6.mean(), ms(c6)[1], c4.mean(), ms(c4)[1])))]
print(by_footprint(chk_7b).round(1).to_string())

**Result: partially reproduced.** The CT26 6-burst mean (4847 +/- 725 a.u., n = 3) lies below the line through 1 to 4 bursts (5719 at 6 bursts) but inside its 95% band (4028 to 7410 on experiments; 3204 to 8225 on group means). It is not above the 4-burst mean (3869 +/- 55) by a gap larger than the error bars, so reading (b) holds and reading (a) does not. CHO: 6-burst mean 3045 against 3194 from its 1-to-4-burst line, inside that band (2526 to 3861).

Under the footprint readings (i) to (iv) the 6-burst mean (4617 to 4845 a.u.) lies below the line (5425 to 5676) but inside its band, and is not above the 4-burst mean by a gap: partially reproduced under every footprint.

### fig7cf-fwhm-with-bursts

> The FWHM of the central peak increased in proportion to the number of bursts in CT26 cells (slope p < 0.001) but not in CHO cells (p = 0.44).

Per-experiment FWHM against bursts; CHO 1-burst wells left out (Fig. 7 legend); CT26 6-burst kept, left out as a variant. Readings O1, O3 and O4.

In [ ]:
rows = []
FW7 = {}
for roi in ROIS:
    for hr in RANGES:
        r = dict(roi=roi, range=hr)
        for cell, bs in (("CT26", ("1", "2", "4", "6")), ("CT26_no6", ("1", "2", "4")), ("CHO", ("2", "4", "6"))):
            cl = cell.split("_")[0]
            fw = {b: fwhm_each(B[(cl, b)], roi, hr) for b in bs}
            FW7[(roi, hr, cell)] = fw
            X = np.concatenate([[int(b)] * len(v) for b, v in fw.items()]); Y = np.concatenate(list(fw.values()))
            ok = np.isfinite(Y) & (Y < 50)
            st = cc.slope_test(X[ok], Y[ok]); sm = cc.slope_test([int(b) for b in bs], [np.nanmean(v) for v in fw.values()])
            r.update({f"{cell}_slope": st["slope"], f"{cell}_p": st["p"], f"{cell}_p_means": sm["p"], f"{cell}_dropped": int((~ok).sum())})
            r.update({f"{cell}_fwhm{b}": np.nanmean(v) for b, v in fw.items()})
        rows.append(r)
R7c = pd.DataFrame(rows); print(R7c.T.to_string(float_format=lambda v: f"{v:.3g}"))

In [ ]:
# Footprint readings (O1 i to iv, claims file as amended 2026-11-20), with the darkest footprint as reference
def chk_7cf():
    rows = []
    for roi in ROIS:
        for hr in RANGES:
            r = dict(roi=roi, range=hr)
            for cell, bs in (("CT26", ("1", "2", "4", "6")), ("CHO", ("2", "4", "6"))):
                fw = {b: fwhm_each(B[(cell, b)], roi, hr) for b in bs}
                X = np.concatenate([[int(b)] * len(v) for b, v in fw.items()]); Y = np.concatenate(list(fw.values())); ok = np.isfinite(Y) & (Y < 50)
                st = cc.slope_test(X[ok], Y[ok]); sm = cc.slope_test([int(b) for b in bs], [np.nanmean(v) for v in fw.values()])
                r.update({f"{cell}_slope": st["slope"], f"{cell}_p": st["p"], f"{cell}_p_means": sm["p"]})
            rows.append(r)
    return rows
T = by_footprint(chk_7cf); print(T.to_string(float_format=lambda v: f"{v:.3g}"))
T["holds"] = (T.CT26_slope > 0) & (T.CT26_p < 0.001) & (T.CHO_p >= 0.05)
print("criterion on experiments per reading, by footprint:", {fp: f"{int(g.holds.sum())} of {len(g)}; CHO p {g.CHO_p.min():.4f} to {g.CHO_p.max():.3f}" for fp, g in T.groupby("footprint")})
q = T[T.footprint != "darkest"]
print(f"readings (i) to (iv), on experiments: CT26 slope positive {bool((q.CT26_slope > 0).all())}, p up to {q.CT26_p.max():.1e}; CHO p {q.CHO_p.min():.3f} to {q.CHO_p.max():.3f}; criterion holds in {int(((q.CT26_slope > 0) & (q.CT26_p < 0.001) & (q.CHO_p >= 0.05)).sum())} of {len(q)}. On group means: CT26 p {q.CT26_p_means.min():.4f} to {q.CT26_p_means.max():.4f}, CHO p {q.CHO_p_means.min():.2f} to {q.CHO_p_means.max():.2f}")

**Result: blocked (methods_underspecified)** (changed from reproduced, resting on an assumption, after the third verification round), as the outcome depends on the background footprint, shown below. With the darkest footprint of each well, the choice made before the reader's answer, on individual experiments the CT26 FWHM slope (1 to 6 bursts) is 0.95 to 1.22 mm per burst with p below 0.001 under every reading, and the CHO slope (2 to 6 bursts) 0.08 to 0.14 mm per burst with p = 0.068 to 0.33, not significant. On group means the CT26 p is 0.0005 to 0.016 and the CHO p 0.44 to 0.64 (paper 0.44). In the 2000 random subsets of the paper's size (printed under fig7be-center-linear-with-bursts), the CT26 slope has p below 0.001 under every reading of O1 and O3 in every draw, and the CHO slope p of at least 0.05 under every reading in 88.0% of draws. The group-means reading would make the claim partially reproduced; it is not adopted because the legend gives p < 0.001 for the CT26 slope of panel B, which is fitted to three group means (one degree of freedom, t of about 636 needed), so the panel B p-value points to per-experiment points. This argument holds for panel B only: with four group means the CT26 FWHM slope of panel C reaches p = 0.0005 under one reading (contiguous ROIs, 4-mm range).

Under the footprint readings (i) to (iv) of the amended O1, the CT26 slope is positive with p at most 5.8e-07 under every reading, but the CHO slope p depends on the footprint: 0.076 to 0.237 with (i), 0.131 to 0.410 with (ii), 0.0065 to 0.020 with (iii) and 0.018 to 0.075 with (iv). The criterion holds in all six readings of ROI and fit range with (i) and (ii), in none with (iii) and in 2 of 6 with (iv); with (iii) and most of (iv) the CHO slope has p below 0.05, which the criterion calls not reproduced. The outcome depends on a footprint the paper does not name, and no position is better supported than another. The claim is blocked (methods_underspecified). In wells located from the YP spot (for example 4 of 8 CHO wells at 2 bursts) the footprint ROIs sit on the nominal square and may miss the real footprint, which bears on how reliable each footprint reading is.

### fig7-sharpest-peak-one-burst

> One burst (x4) in CT26 cells gave the sharpest peak of the study, FWHM 1.1 +/- 0.1 mm, 7.3% of the inter-electrode gap and 2.3% of the electrode array area.

Area readings: (a) disk of diameter FWHM over the square with 15.3-mm diagonals (117 mm2); (b) disk of radius FWHM over the square of the electrode centres (12.8-mm sides); (c) disk of diameter FWHM over the square of the electrode centres. The comparison groups are every group fitted per experiment in 001276 and 001277 (the Fig. 4C groups are not in any dandiset).

In [ ]:
for roi in ROIS:
    for hr in RANGES:
        v = fwhm_each(B[("CT26", "1")], roi, hr); m, se, n = ms(v)
        A_a = np.pi * (m / 2) ** 2 / (cc.DIAG_GAP_MM ** 2 / 2); A_b = np.pi * m ** 2 / cc.SIDE_MM ** 2; A_c = np.pi * (m / 2) ** 2 / cc.SIDE_MM ** 2
        groups = {f"001276 {c} {b}": np.nanmean(fwhm_each(B[(c, b)], roi, hr)) for c in ("CT26", "CHO") for b in ("1", "2", "4", "6")}
        groups.update({f"001277 {d} ns": np.nanmean(fwhm_each(g, roi, hr)) for d, g in f6_main.items()})
        groups.pop("001276 CHO 1")
        smallest = min(groups, key=groups.get)
        print(f"ROI {roi}, range {hr}: CT26 1 burst FWHM {m:.3f} +/- {se:.3f} mm (n={n}), {100 * m / 15.3:.2f}% of 15.3 mm; area (a) {100 * A_a:.2f}%, (b) {100 * A_b:.2f}%, (c) {100 * A_c:.2f}%; smallest group: {smallest} ({groups[smallest]:.3f}); next: {sorted(groups.values())[1]:.3f}")
print("per well (ROI a, 4 mm):", [(o["subject"], round(cc.fit(*prof(o), "lorentz", 4.0)["fwhm"], 3)) for o in B[("CT26", "1")]])
print("CHO 1 burst (left out of the comparison, Fig. 7 legend):", np.round(fwhm_each(B[("CHO", "1")], "a", 4.0), 2).tolist())

In [ ]:
# The Fig. 5 YP groups (001284, reported by the criterion), and the sensitivity to locating the array
# (problem B4 of round 1): the CT26 1-burst wells measured from footprints alone, and spot wells
# shifted by the median paired FWHM difference.
for roi in ROIS:
    for hr in RANGES:
        f5 = {f"001284 YP {e} mm": np.nanmean(fwhm_each(g, roi, hr)) for e, g in yp.items()}
        g1 = B[("CT26", "1")]; v = fwhm_each(g1, roi, hr); sp = np.array([o["how"] == "spot centre" for o in g1])
        adj = np.where(sp, v - FWHM_BIAS[(roi, hr)], v).mean()
        print(f"ROI {roi}, range {hr}: Fig. 5 groups {{{', '.join(f'{k}: {u:.2f}' for k, u in f5.items())}}}; CT26 1 burst, footprint well only {v[~sp].mean():.3f} mm (n={int((~sp).sum())}), spot wells adjusted {adj:.3f} mm, {100 * adj / 15.3:.2f}%, area (b) {100 * np.pi * adj ** 2 / cc.SIDE_MM ** 2:.2f}%")

In [ ]:
# Footprint readings (O1 i to iv, claims file as amended 2026-11-20), with the darkest footprint as reference
def chk_7s():
    rows = []
    for roi in ROIS:
        for hr in RANGES:
            m, se, n = ms(fwhm_each(B[("CT26", "1")], roi, hr))
            groups = {f"001276 {c} {b}": np.nanmean(fwhm_each(B[(c, b)], roi, hr)) for c in ("CT26", "CHO") for b in ("1", "2", "4", "6") if (c, b) != ("CHO", "1")}
            groups.update({f"001277 {d} ns": np.nanmean(fwhm_each(g, roi, hr)) for d, g in f6_main.items()})
            groups.update({f"001284 YP {e} mm": np.nanmean(fwhm_each(g, roi, hr)) for e, g in yp.items()})
            area_b = 100 * np.pi * m ** 2 / cc.SIDE_MM ** 2
            rows.append(dict(roi=roi, range=hr, fwhm=m, se=se, pct=100 * m / 15.3, area_b=area_b, smallest=min(groups, key=groups.get), next_fwhm=sorted(groups.values())[1],
                             all_three=bool(abs(m - 1.1) <= 0.1 and abs(100 * m / 15.3 - 7.3) <= 0.73 and abs(area_b - 2.3) <= 0.23)))
    return rows
T = by_footprint(chk_7s); print(T.round(3).to_string())
q = T[T.footprint != "darkest"]
print(f"readings (i) to (iv): FWHM {q.fwhm.min():.3f} to {q.fwhm.max():.3f} mm; all three values within tolerance in {int(q.all_three.sum())} of {len(q)}; CT26 1 burst the smallest group in {int((q.smallest == '001276 CT26 1').sum())} of {len(q)}")

**Result: reproduced.** CT26 after 1 burst (x4): FWHM 1.031 to 1.134 mm (s.e. 0.15 to 0.17, n = 3), 6.74 to 7.41% of 15.3 mm; as a disk of radius FWHM over the square of electrode centres (area reading b) 2.04 to 2.47% of the array, inside 2.3 +/- 0.23% under contiguous ROIs (2.41 to 2.47%). All three values are within tolerance under contiguous ROIs, and the FWHM is the smallest group mean in 001276 and 001277 under every reading (next: 900 ns, 1.69 to 1.77 mm). The Fig. 4C groups are in no dandiset. The Fig. 5 YP groups of 001284, reported by the criterion, have mean FWHM 1.72 to 3.57 mm, all larger. Two of the three 1-burst wells were located from the YP spot; with their FWHM shifted by the median difference between the two ways of locating the array, the 1-burst FWHM is 1.117 to 1.124 mm (7.30 to 7.35%, area reading b 2.39 to 2.42%) under whole-mm ROIs, still inside all three tolerances.

Under the footprint readings (i) to (iv) the CT26 1-burst FWHM is 0.932 to 1.126 mm, all three values are within tolerance in 12 of 24 combinations, and it is the smallest group in all 24.

### fig7-offtarget-40-50-fold

> In the burst-number experiments with MHz-CANCAN, YP emission near the electrodes was 40 to 50 times smaller than at the remote target in the centre.

Ratio of centre YP to YP at 6 or 7 mm (O5 a, b), as the ratio of group means (i) or the mean of per-experiment ratios (ii), under O1; groups (A) CT26 1 burst, (B) CT26 1, 2 and 4 bursts, (C) every CHO burst number. Background-subtracted values; a near-electrode value near zero makes a per-experiment ratio unstable, so (ii) is also given as the median.

In [ ]:
rows = []
for cell in ("CT26", "CHO"):
    for b in ("1", "2", "4", "6"):
        g = B[(cell, b)]
        for roi in ROIS:
            c0 = np.array([at(o, 0, roi) for o in g])
            for d in (6, 7):
                y = np.array([at(o, d, roi) for o in g])
                rows.append(dict(cell=cell, bursts=b, roi=roi, dist=d, n=len(g), ratio_of_means=c0.mean() / y.mean(), mean_of_ratios=np.mean(c0 / y), median_ratio=np.median(c0 / y), near_mean=y.mean(), near_se=ms(y)[1], centre=c0.mean()))
R7o = pd.DataFrame(rows); R7o["in_36_55_i"] = R7o.ratio_of_means.between(36, 55); R7o["in_36_55_ii"] = R7o.mean_of_ratios.between(36, 55)
print(R7o.round(2).to_string())
# background variant: the mean of the four footprints instead of the darkest one
for b in ("1", "2"):
    g = B[("CT26", b)]
    for d in (6, 7):
        c0 = np.array([o["pa"]["raw"][:, 0].mean() - o["pa"]["backgrounds"].mean() for o in g])
        i = int(np.argmin(np.abs(cc.DIST_A - d)))
        y = np.array([o["pa"]["raw"][:, i].mean() - o["pa"]["backgrounds"].mean() for o in g])
        print(f"variant, background = mean of four footprints, CT26 {b} burst, {d} mm: ratio of means {c0.mean() / y.mean():.1f}, near-electrode mean {y.mean():.1f} a.u.")
print("background used (a.u.), CT26 1 burst:", [round(o["pa"]["background"]) for o in B[("CT26", "1")]], "all four footprints:", [np.round(o["pa"]["backgrounds"]).tolist() for o in B[("CT26", "1")]])

Background from each footprint. The paper subtracts the mean of 3 ROIs "inside an electrode footprint" (Methods 2.5) without saying which. As the near-electrode signal is a few tens of a.u. above background, the choice changes the ratio. The claims file, as amended on 2026-11-20 in answer to this attempt's question, makes the footprint an open reading (O1 i to iv, the footprint at the same position in every well) and this criterion runs each one, the outcome resting on the footprint that gives it. Below: each footprint at a fixed position in the image (the O1 readings), and as variants each footprint ranked by value (0 darkest, 3 brightest) and the mean of the four.

In [ ]:
rows = []
for cell in ("CT26", "CHO"):
    for b in ("1", "2", "4", "6"):
        g = B[(cell, b)]
        for bg in (0, 1, 2, 3, "mean"):
            for roi in ROIS:
                c0 = np.array([at_bg(o, 0, bg, roi) for o in g])
                for d in (6, 7):
                    y = np.array([at_bg(o, d, bg, roi) for o in g])
                    rows.append(dict(cell=cell, bursts=b, background=bg, roi=roi, dist=d, near_mean=y.mean(), ratio_of_means=c0.mean() / y.mean(), mean_of_ratios=np.mean(c0 / y)))
R7bg = pd.DataFrame(rows)
R7bg["in_i"] = R7bg.ratio_of_means.between(36, 55); R7bg["in_ii"] = R7bg.mean_of_ratios.between(36, 55)
print(R7bg.round(1).to_string())
print("spread of the four footprints within a well (max - min, a.u.), by group:")
print({k: round(float(np.median([np.ptp(o["pa"]["backgrounds"]) for o in v])), 1) for k, v in B.items()})
def readings_hit(sel):
    q = R7bg[sel]
    return q[(q.in_i) | (q.in_ii)][["cell", "bursts", "background", "roi", "dist", "ratio_of_means", "mean_of_ratios"]].round(1)
print("reading (A), CT26 1 burst, combinations inside 36 to 55:"); print(readings_hit((R7bg.cell == "CT26") & (R7bg.bursts == "1")).to_string())
for lab, cl, bs in (("(B) CT26 1, 2 and 4 bursts", "CT26", ("1", "2", "4")), ("(C) CHO 1, 2, 4 and 6 bursts", "CHO", ("1", "2", "4", "6"))):
    q = R7bg[(R7bg.cell == cl) & R7bg.bursts.isin(bs)]
    for meas in ("i", "ii"):
        ok = q.groupby(["background", "roi", "dist"])[f"in_{meas}"].all()
        print(f"reading {lab}, ratio ({meas}): combinations with every group inside 36 to 55: {ok[ok].index.tolist()}")
# The same with the footprint at a fixed position in the image (corner order of cc.find_array: upper
# left, upper right, lower right, lower left), a choice that does not depend on the values
def at_pos(o, d, k, roi="a"):
    p = o["pa"] if roi == "a" else o["pb"]
    i = int(np.argmin(np.abs(p["dists"] - d)))
    return float(p["raw"][:, i].mean() - p["backgrounds"][k])
rows = []
for cell in ("CT26", "CHO"):
    for b in ("1", "2", "4", "6"):
        g = B[(cell, b)]
        for k in range(4):
            for roi in ROIS:
                c0 = np.array([at_pos(o, 0, k, roi) for o in g])
                for d in (6, 7):
                    y = np.array([at_pos(o, d, k, roi) for o in g])
                    rows.append(dict(cell=cell, bursts=b, corner=k, roi=roi, dist=d, ratio_of_means=c0.mean() / y.mean(), mean_of_ratios=np.mean(c0 / y)))
R7pos = pd.DataFrame(rows); R7pos["in_i"] = R7pos.ratio_of_means.between(36, 55); R7pos["in_ii"] = R7pos.mean_of_ratios.between(36, 55)
print("reading (A), CT26 1 burst, background from the footprint at a fixed corner:")
print(R7pos[(R7pos.cell == "CT26") & (R7pos.bursts == "1")].round(1).to_string())
for lab, cl, bs in (("(B)", "CT26", ("1", "2", "4")), ("(C)", "CHO", ("1", "2", "4", "6"))):
    q = R7pos[(R7pos.cell == cl) & R7pos.bursts.isin(bs)]
    for meas in ("i", "ii"):
        ok = q.groupby(["corner", "roi", "dist"])[f"in_{meas}"].all()
        print(f"reading {lab}, ratio ({meas}), fixed corner: combinations with every group inside 36 to 55: {ok[ok].index.tolist()}")

**Result: reproduced, resting on an assumption.** With the darkest footprint as background, the choice used elsewhere, no reading puts its groups inside 36 to 55: CT26 1 burst 13.5 to 25.0 (ratio of means) and 13.2 to 25.5 (mean of ratios); CT26 2 bursts 11.7 to 28.4 and 15.7 to 36.7; 4 bursts 2.8 to 8.0; CHO groups 1.5 to 236. The near-electrode signal is a few tens of a.u. above background, and the four footprints of a CT26 1-burst well differ by a median of 31 a.u., so the ratio depends on which footprint gives the background, which the paper does not say. Taking the footprint at a fixed position, reading (A) at 6 mm gives 42.4 (ratio of means) and 45.4 (mean of ratios) with the upper-left footprint and whole-mm ROIs, and three of the four positions give a value inside 36 to 55 under at least one ROI reading; the upper-right one gives 24.8 to 29.8. At 7 mm every choice gives 13.9 to 23.9. Readings (B) and (C) have no combination with every group inside the range under any background. The criterion needs one combination, which reading (A) at 6 mm meets. The claims file, amended on 2026-11-20 in answer to this attempt's question, now makes the footprint an open reading (O1 i to iv, the footprint at the same position in every well) and says the outcome rests on the footprint that gives the value, as an assumption; that is how it is recorded. The darkest footprint gives the lowest ratio.

### fig7-cho-replicates-ct26

> Remote focusing of electroporation was consistent across the two cell types; in CHO cells as in CT26 cells the YP uptake peaked at the centre of the array.

In [ ]:
rows = []
for cell in ("CHO", "CT26"):
    for b in ("1", "2", "4", "6"):
        g = B[(cell, b)]
        for roi in ROIS:
            c0 = np.array([at(o, 0, roi) for o in g]); m0, s0, n = ms(c0)
            r = dict(cell=cell, bursts=b, roi=roi, n=n, centre=m0, se=s0)
            for d in (6, 7):
                y = np.array([at(o, d, roi) for o in g]); m, s, _ = ms(y)
                r[f"y{d}"] = m; r[f"se{d}"] = s; r[f"gap{d}"] = (m0 > m) and cc.gap_exceeds(m0, s0, m, s)
            rows.append(r)
R7x = pd.DataFrame(rows); print(R7x.round(1).to_string())
q = R7x[R7x.bursts.isin(["2", "4", "6"])]
print("bursts 2 to 6, every cell line and reading, centre above 6 and 7 mm by a gap:", bool((q.gap6 & q.gap7).all()))

In [ ]:
# Footprint readings (O1 i to iv, claims file as amended 2026-11-20), with the darkest footprint as reference
def chk_7x():
    rows = []
    for cell in ("CHO", "CT26"):
        for b in ("1", "2", "4", "6"):
            g = B[(cell, b)]
            for roi in ROIS:
                c0 = np.array([at(o, 0, roi) for o in g]); m0, s0, n = ms(c0)
                r = dict(cell=cell, bursts=b, roi=roi, centre=m0)
                for d in (6, 7):
                    y = np.array([at(o, d, roi) for o in g]); m, s, _ = ms(y)
                    r[f"y{d}"] = m; r[f"gap{d}"] = bool(m0 > m and cc.gap_exceeds(m0, s0, m, s))
                rows.append(r)
    return rows
T = by_footprint(chk_7x)
q = T[T.footprint != "darkest"]
print(q.round(1).to_string())
print("readings (i) to (iv), bursts 2 to 6, both cell lines: centre above 6 and 7 mm by a gap in", int((q[q.bursts != "1"].gap6 & q[q.bursts != "1"].gap7).sum()), "of", int((q.bursts != "1").sum()), "; 1 burst:", int((q[q.bursts == "1"].gap6 & q[q.bursts == "1"].gap7).sum()), "of", int((q.bursts == "1").sum()))

**Result: reproduced.** For 2, 4 and 6 bursts in both cell lines, mean centre YP exceeds mean YP at 6 and at 7 mm by more than the error bars under both ROI readings (CHO centre 578 to 3045 a.u., near the electrodes 8 to 537 a.u.). One burst also holds in both lines.

Under the footprint readings (i) to (iv) the gaps hold in all 48 combinations for 2 to 6 bursts, and at 1 burst in 14 of 16.

## Summary

| Claim | Outcome |
|---|---|
| fig1e-field-simulation | blocked: data_missing |
| fig2-heating-below-44c | blocked: data_missing |
| fig3a-centre-enhanced | partially reproduced (on 001276, two rates held) |
| fig3bc-rate-suppresses-off-target | blocked: data_missing; reproduced (assumed, pooling b) proposed, the verifier did not agree (would give blocked: methods_underspecified) |
| fig3-lorentzian-profile | partially reproduced (on 001276, two rates held) |
| fig3-tail-deviation-falls-with-rate | blocked: data_missing |
| fig4b-40-fold-ratio | reproduced (assumed, background footprint) |
| fig4b-centre-no-rate-effect | blocked: data_missing |
| fig4b-near-electrode-decline | blocked: data_missing |
| fig4c-fwhm-falls-with-rate | blocked: data_missing |
| fig4c-log-trend | blocked: data_missing |
| fig5-raised-electrodes-weaken-centre | not reproduced |
| fig5-killing-focused-centre | reproduced |
| fig5ab-yp-spot-slightly-larger-than-pr | not reproduced |
| fig5b-gaussian-pr-lorentzian-yp | reproduced |
| fig5b-reversible-beyond-2mm | reproduced |
| fig5c-nuclear-yp-centre | partially reproduced |
| fig6b-peak-900ns-not-significant | reproduced |
| fig6c-normalized-decline-similar | not reproduced |
| fig6d-longer-pulses-sharper-peak | not reproduced |
| fig6d-600-900-differ-from-300 | blocked: methods_underspecified (the outcome depends on the background footprint) |
| fig6d-900ns-fwhm | reproduced |
| fig7be-center-linear-with-bursts | partially reproduced |
| fig7b-ct26-saturation | partially reproduced |
| fig7cf-fwhm-with-bursts | blocked: methods_underspecified (the outcome depends on the background footprint) |
| fig7-sharpest-peak-one-burst | reproduced |
| fig7-offtarget-40-50-fold | reproduced (assumed) |
| fig7-cho-replicates-ct26 | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The ROIs are placed by code on images reduced to 26-um blocks, at whole-mm distances or in contiguous steps (the paper does not give its positions), along diagonals found from the electrode footprints. Where the footprints could not be found (most 1- and 2-mm wells, and wells with faint footprints), the centre is the centre of the central YP spot, with a fixed rotation; this is an assumption. In wells where both are known the two centres are 0.36 mm apart (median), and measuring from the spot raises the centre value by a median of 7% and narrows the FWHM by a median of 0.10 to 0.13 mm; fig6d-900ns-fwhm and fig7-sharpest-peak-one-burst are also shown with this difference taken out; fig6d-longer-pulses-sharper-peak, fig6d-600-900-differ-from-300 and fig7cf-fwhm-with-bursts are not.
- Background is the mean of 3 ROIs in one footprint. The claims file, amended on 2026-11-20 in answer to this attempt's question, makes the footprint an open reading (O1, readings i to iv, the footprint at the same position in every well). Each claim's first cells use the darkest footprint of each well, the choice made before the answer, and every claim whose criterion cites O1 is also run under each of (i) to (iv). The footprint changes the outcome of fig6d-600-900-differ-from-300 and fig7cf-fwhm-with-bursts, which are blocked as methods_underspecified; the ratio claims rest on the footprint that gives their value. In wells located from the YP spot the footprint ROIs are placed on the nominal square, so they may fall partly outside the real footprint.
- The parallel ROIs of fig3bc-rate-suppresses-off-target are placed by code on the whole-well images, from the centre to the middle of each side.
- The dandisets hold more wells than the paper reports for several groups (001284 YP, 001276 CHO); all are used, and Fig. 5 is also shown with the August 2024 YP wells (the paper's n) and Fig. 7 with random subsets of the paper's size. In 001277 the main cells leave out two 600-ns wells (P1-20240926-B2 and B3, both on the one plate that holds three 600-ns wells) so that n = 5 as in the paper. B2's YP spot is 5.4 mm from the array centre, but B3 differs from the other wells only in its broad profile, the quantity fig6d-longer-pulses-sharper-peak tests. Every claim on Figure 6 is also shown with all seven and with only B2 left out; fig6d-longer-pulses-sharper-peak holds for none of these nor for the other 20 ways of choosing five of the seven, and is not reproduced.
- Per-experiment fits are used as the points of the regressions of Figures 6D and 7 (claims file O4, reading a); the group-mean reading is shown. With the darkest footprint it would change the outcome of fig7cf-fwhm-with-bursts; it does not change the outcome of fig6d-longer-pulses-sharper-peak or fig7be-center-linear-with-bursts.

What was found about the datasets:

- DANDI:001047 holds images from a 3.8-mm array with a 3.2-kV first pulse, not the paper's 15.3-mm array. Of the packet-rate series of Figures 3 and 4, only 1 Hz, 0.2 MHz and controls are deposited, in 001276; the 1-kHz and 10-kHz wells are in none of the four dandisets. No dandiset holds the field simulation, the thermochromic film images, or the high-resolution images of Figure 5C.
- 001276 holds CANCAN wells at 1 Hz and control wells that its description does not mention, and more CHO wells than the paper's n = 4. Its CT26 wells at 2 bursts (x4), CANCAN at 1 Hz and 0.2 MHz and controls, fit the packet-rate series of Figures 3 and 4 by the paper's protocol; after a reader's answer the claims file assigns four of its claims to them.
- 001277: per-file metadata give 9 packets at 0.2 MHz for every pulse duration; of the 600-ns wells, P1-20240926-B2 has its YP spot 5.4 mm from the array centre and P1-20240926-B3 far more uptake 3 to 5 mm out than any other well of the dandiset; one well is a sham exposure; the Cy3-DAPI image of well P2-20240925-B3 is deposited twice.
- 001284: the FITC image of well P1-20240828-A3 is almost empty (11 MB, 98% zero pixels); YP groups hold 12, 11 and 11 wells against the paper's 6 to 7; two Hoechst files state 900-ns pulses in wells whose other files state 600 ns.
- The files give a 10-mm inter-electrode distance, which is none of the distances the paper gives (15.3-mm diagonal between needle edges, 12.8 mm between centres along a side).

In [ ]:
print(f"total run time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")